# CLDT-Thread — Phase 3: Project Frame dan Raw Evidence End-to-End
## Implementasi Protocol Framing, Gateway Bridge, dan Host Recorder

**Technical Source Baseline:** commit `0dee73247d4e87a3afec8fb90f37439cbb96523b`  
**Notebook Revision:** Reviewed against repository HEAD. Commits following the source baseline refine documentation and rendering; all underlying project source code, headers, schemas, and CMake contracts remain identical to the baseline commit.

Panduan implementasi Week 3 berdasarkan file pada commit baseline `0dee73247d4e87a3afec8fb90f37439cbb96523b`. Snippet mempertahankan header, source, build, Python, dan manifest repo tanpa adapter notebook. Kolom **e.g.** memperlihatkan format bukti yang diisi dari hasil pilot nyata.

| Area Lingkup | Sasaran dan Batasan Minimum |
|---|---|
| **Target Data Path** | Endpoint mengodekan satu project frame, mengirimkannya melalui Thread, lalu gateway memindahkan datagram ke observation path melalui dua queue yang bounded dan terukur. |
| **Target Evidence Path** | Host menerima raw broker bytes, membekukan ready manifest, menulis `events.ndjson`, lalu menghasilkan item audit dan aggregate reconciliation yang dapat diulang. |
| **Target Closure** | Closure sah setelah raw lifecycle lulus item audit dan aggregate reconciliation pada baseline berulang. Prediction, fidelity gate, command, dan policy tetap ditunda. |

## Peta Kerja Week 3
### Aliran Data, Raw Evidence Path, dan Titik Rekonsiliasi

Diagram ini memperlihatkan urutan ownership dari release di endpoint sampai replay di host. Setiap queue dan file adalah boundary yang harus mempertahankan identity `run_id`, `node_id`, `boot_id`, dan `sequence`; model maupun command path tidak ikut pada jalur Week 3.

![Phase 3 implementation roadmap](docs/diagrams/phase3/implementation-roadmap.svg)

[Mermaid source](docs/diagrams/phase3/implementation-roadmap.mmd)

## Step 1: Entry Gate Week 2
### Verifikasi Kesiapan Baseline dan Bukti Masuk

Week 3 boleh dimulai sebagai pekerjaan source, tetapi baseline fisik end-to-end baru dijalankan setelah queue ownership, local terminal, dan dua lapis reconciliation Week 2 benar-benar jelas. Nilai berikut disalin dari notebook sebelumnya; contoh di kolom kanan tidak menyatakan hardware sudah lulus.

| Bukti masuk | Contoh format — ganti dengan hasil aktual |
|---|---|
| Verdict one-endpoint UDP gate | e.g. PASS |
| Repetisi cold boot valid | e.g. 3/3 |
| Aggregate UDP delivery ratio | e.g. 0.9989 |
| Reset/detach tak terduga | e.g. 0 / 0 |
| `cldt_event_trace_t.dropped_records` pada local pilot | e.g. 0 |
| `cldt_item_audit_t.consistent` | e.g. true |
| Semua elemen `cldt_reconciliation_t.consistent` | e.g. true untuk setiap traffic class yang dipakai |
| Endpoint A dan B attached | e.g. PASS; actual role, parent, RLOC16, partition, dan IPv6 tersimpan |
| `setup.thread_channel` | e.g. 15 |
| Source commit | e.g. 40 karakter hexadecimal |
| SHA-256 binary gateway dan dua endpoint | e.g. tiga digest, masing-masing 64 karakter hexadecimal |
| Lokasi raw log Week 2 | e.g. C:\logs\cldt\week-02 |

Urutan masuknya:

1. Hasil local-only diperiksa lebih dahulu. Jika representasi `CLDT_EVENT_TASK_FINISH` pada `cldt_metrics_t` masih belum selesai, aggregate local gate belum sah.
2. Contract full-queue pada `cldt_deadline_queue_acquire()` dan `cldt_deadline_queue_commit()` harus benar-benar dapat dijalankan; komentar source saat ini belum cukup.
3. Endpoint A dan B memakai binary/configuration yang identitasnya sudah dicatat.
4. Project frame baru masuk setelah test protocol host mempunyai fixed vectors. UDP upstream tetap menjadi jalur pembanding bring-up.
5. Baseline Week 3 tidak memakai host model dan tidak menerima remote actuation.

## Step 2: Batasan Komponen dan Kepemilikan File
### Pemetaan Modul Aktif dan Lingkup yang Ditunda pada Week 3

| Hasil Week 3 | File/fungsi repo yang memegang pekerjaan | Tidak ikut |
|---|---|---|
| Project frame | `common/src/cldt_protocol.c`, `tests/test_protocol.c`, dan CRC dari Week 1 | Model dan policy |
| Endpoint UDP adapter | `firmware/endpoint/main/thread_transport.c` serta observation subset `firmware/endpoint/main/endpoint_runtime.c` | `cldt_endpoint_runtime_receive_command()` |
| Credential prerequisite | `firmware/gateway/main/gateway_provisioning.c` | Menaruh secret di source/notebook |
| Bounded Thread receive | `firmware/gateway/main/thread_bridge.c` dan `cldt_gateway_runtime_t.thread_rx_queue` | Buffer tanpa batas |
| MAC/role observation | `firmware/gateway/main/thread_diagnostic.c` | Mengubah actual role menjadi role yang diharapkan |
| Clock mapping dan uncertainty | `common/src/cldt_clock_sync.c` dan `tests/test_clock_sync.c` | Klaim one-way latency saat mapping invalid |
| Observation publication | `firmware/gateway/main/backhaul.c` dan `cldt_gateway_runtime_t.observation_queue` | Retained observation sebagai current truth |
| Shadow-safe gateway run | init/begin/end subset `firmware/gateway/main/policy_guard.c` | `cldt_policy_guard_accept()` dan remote fallback behavior |
| Manifest/profile admission | `host/experiment_config.c`, `common/src/cldt_control_profile.c`, `tests/test_control_profile.c`, dan `host/main.c` | Menerima `state: "template"` atau profile identity tanpa digest |
| Raw recorder | `host/run_recorder.c` | Mengurutkan atau menghapus raw event |
| Broker receive | `host/broker_io.c` | Model update di callback |
| Baseline coordinator path | `host/coordinator.c` dengan `host_model_enabled == false` dan `remote_actuation_enabled == false` | Kalman, twin model, fidelity gate, proposal |
| Item audit/reconciliation | `cldt_metrics_audit_sorted_trace()`, `cldt_metrics_reconcile()`, dan lifecycle subset `host/analysis/reproduce.py` | Scoring tiga model |

Sembilan contract gap perlu ditutup secara eksplisit; notebook tidak memberi nama field atau format baru untuk menyamarkannya:

1. `firmware/gateway/main/CMakeLists.txt` belum memasukkan `thread_diagnostic.c` ke `SRCS`.
2. `cldt_backhaul_publish_observation()` meminta envelope metadata, tetapi repository belum mendefinisikan byte contract atau topic contract untuk envelope tersebut.
3. `cldt_run_recorder_append()` meminta NDJSON, tetapi exact key set dan schema event line belum didefinisikan.
4. `host/experiment_config.c` meminta maintained JSON parser, schema validation, duplicate-key rejection, dan canonical digest, tetapi dependency tersebut belum dipilih pada `host/CMakeLists.txt`.
5. `cldt_clock_sync.c` belum membekukan convergence, delay-outlier, drift, uncertainty, dan reset rules yang menentukan apakah one-way time boleh dipakai.
6. `host/main.c` meminta resolved profile dari versioned local registry, sedangkan tree belum memiliki registry document/format. Notebook tidak mengganti contract itu dengan string profile fiktif.
7. `host/broker_io.c` meminta koneksi/subscription/publish MQTT, tetapi `host/CMakeLists.txt` belum memilih atau menautkan client library host.
8. `host/main.c` meminta durable global run ledger dan CSPRNG reservation, tetapi tree belum menetapkan owner module, record format, atau storage path ledger.
9. `cldt_clock_sync.c` mempunyai in-memory exchange, tetapi repo belum mempunyai wire payload dan endpoint/gateway call path untuk menghasilkan physical four-timestamp exchange.

Kesembilannya adalah pekerjaan contract sebelum implementasi. Nilai/topic/key buatan notebook tidak boleh dianggap solusi. Untuk Week 3, contract dipilih sekecil mungkin, ditulis di source/header yang menjadi pemiliknya, diberi fixed test, lalu dibekukan sebelum pilot. Bila profile registry atau durable ledger belum mempunyai artefak yang nyata, ready baseline tetap blocked; notebook tidak mengarang lokasi atau formatnya.

## Step 3: Protocol Framing dan Codec
### Serialisasi dan Deserialisasi Format Frame CLDT (cldt_protocol.h)

### Header contract: `common/include/cldt/cldt_protocol.h`

In [ ]:
%%writefile /content/cldt_scratch/cldt_protocol.h
#ifndef CLDT_PROTOCOL_H
#define CLDT_PROTOCOL_H

#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef cldt_status_t (*cldt_authenticate_fn)(
    void *context,
    const uint8_t *bytes,
    size_t byte_count,
    uint8_t output_tag[CLDT_AUTH_TAG_BYTES]);

typedef struct {
    cldt_authenticate_fn calculate_tag;
    void *context;
} cldt_authenticator_t;

/*
 * Version 1 frame layout. Every multi-byte integer uses network byte order.
 * The two reserved bytes are transmitted as zero and rejected when nonzero.
 * These offsets are the wire contract; sizeof(cldt_frame_meta_t) is not.
 */
#define CLDT_WIRE_MAGIC_OFFSET 0U
#define CLDT_WIRE_VERSION_OFFSET 2U
#define CLDT_WIRE_KIND_OFFSET 3U
#define CLDT_WIRE_TRAFFIC_CLASS_OFFSET 4U
#define CLDT_WIRE_FLAGS_OFFSET 5U
#define CLDT_WIRE_HOP_LIMIT_OFFSET 7U
#define CLDT_WIRE_NODE_ID_OFFSET 8U
#define CLDT_WIRE_BOOT_ID_OFFSET 12U
#define CLDT_WIRE_SEQUENCE_OFFSET 16U
#define CLDT_WIRE_POLICY_EPOCH_OFFSET 20U
#define CLDT_WIRE_RUN_ID_OFFSET 24U
#define CLDT_WIRE_TRANSMIT_LOCAL_US_OFFSET 32U
#define CLDT_WIRE_DEADLINE_LOCAL_US_OFFSET 40U
#define CLDT_WIRE_PAYLOAD_BYTES_OFFSET 48U
#define CLDT_WIRE_RESERVED_OFFSET 50U
#define CLDT_WIRE_RESERVED_BYTES 2U
#define CLDT_WIRE_CRC32C_OFFSET 52U
#define CLDT_WIRE_AUTH_TAG_OFFSET 56U

/*
 * Version 1 policy payload layout. Array elements are contiguous and encoded
 * in traffic-class order from CLDT_TRAFFIC_CONTROL through CLDT_TRAFFIC_BULK.
 * The policy epoch in this payload must equal the frame metadata epoch.
 */
#define CLDT_POLICY_WIRE_RELEASE_PERIOD_OFFSET 0U
#define CLDT_POLICY_WIRE_PHASE_OFFSET 16U
#define CLDT_POLICY_WIRE_BURST_LIMIT_OFFSET 32U
#define CLDT_POLICY_WIRE_BATCH_SIZE_OFFSET 40U
#define CLDT_POLICY_WIRE_TOKEN_RATE_OFFSET 48U
#define CLDT_POLICY_WIRE_EPOCH_OFFSET 64U
#define CLDT_POLICY_WIRE_ISSUED_GATEWAY_US_OFFSET 68U
#define CLDT_POLICY_WIRE_TTL_MS_OFFSET 76U
#define CLDT_POLICY_WIRE_BYTES 80U

#if (CLDT_WIRE_AUTH_TAG_OFFSET + CLDT_AUTH_TAG_BYTES) != CLDT_WIRE_HEADER_BYTES
#error "Version 1 frame offsets do not match CLDT_WIRE_HEADER_BYTES"
#endif

#if CLDT_POLICY_STREAM_COUNT != 4U
#error "Version 1 policy layout requires exactly four traffic classes"
#endif

#if (CLDT_POLICY_WIRE_TTL_MS_OFFSET + 4U) != CLDT_POLICY_WIRE_BYTES
#error "Version 1 policy offsets do not match CLDT_POLICY_WIRE_BYTES"
#endif

/*
 * Returns the exact output size required for this payload. Zero means the
 * payload cannot be represented by the current protocol version.
 */
size_t cldt_protocol_encoded_size(size_t payload_bytes);

/*
 * Encodes one frame into caller-owned storage. No heap allocation or I/O is
 * permitted. output_bytes is written only on success.
 */
cldt_status_t cldt_protocol_encode(
    const cldt_frame_meta_t *meta,
    const uint8_t *payload,
    size_t payload_bytes,
    const cldt_authenticator_t *authenticator,
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes);

/*
 * Decodes and validates one complete datagram. The returned payload view
 * borrows memory from input. The function must reject trailing bytes,
 * truncation, an unsupported version, CRC failure, and required-auth failure.
 */
cldt_status_t cldt_protocol_decode(
    const uint8_t *input,
    size_t input_bytes,
    const cldt_authenticator_t *authenticator,
    bool authentication_required,
    cldt_frame_view_t *output_view);

/*
 * Applies freshness and ordering checks after successful decoding. Times are
 * in the gateway monotonic domain; uncertainty expands the rejection margin.
 * On an endpoint, applied_epoch must be the RAM mirror of a valid durable
 * replay record. The caller still owns issuer validation, durable advancement,
 * local limits, and atomic policy publication.
 */
cldt_status_t cldt_protocol_validate_command(
    const cldt_frame_view_t *frame,
    cldt_run_id_t active_run_id,
    cldt_policy_epoch_t applied_epoch,
    uint64_t now_gateway_us,
    uint32_t time_uncertainty_us);

#ifdef __cplusplus
}
#endif

#endif


### Type contract: `common/include/cldt/cldt_types.h`

In [ ]:
%%writefile /content/cldt_scratch/cldt_types.h
#ifndef CLDT_TYPES_H
#define CLDT_TYPES_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_PROTOCOL_MAGIC UINT16_C(0x434C)
#define CLDT_PROTOCOL_VERSION UINT8_C(1)
#define CLDT_WIRE_HEADER_BYTES UINT16_C(72)
#define CLDT_MAX_PAYLOAD_BYTES UINT16_C(256)
#define CLDT_AUTH_TAG_BYTES 16U
#define CLDT_TRACE_DETAIL_BYTES 24U
#define CLDT_POLICY_STREAM_COUNT 4U
#define CLDT_COMMAND_AUTHORITY_NODE_ID UINT32_C(0)

typedef uint32_t cldt_node_id_t;
typedef uint64_t cldt_run_id_t;
typedef uint32_t cldt_boot_id_t;
typedef uint32_t cldt_sequence_t;
typedef uint32_t cldt_policy_epoch_t;

typedef enum {
    CLDT_NODE_GATEWAY_HOST = 0,
    CLDT_NODE_RADIO_COPROCESSOR,
    CLDT_NODE_ROUTER_ENDPOINT,
    CLDT_NODE_LOW_POWER_ENDPOINT
} cldt_node_role_t;

typedef enum {
    CLDT_TRAFFIC_CONTROL = 0,
    CLDT_TRAFFIC_CRITICAL,
    CLDT_TRAFFIC_TELEMETRY,
    CLDT_TRAFFIC_BULK,
    CLDT_TRAFFIC_COUNT
} cldt_traffic_class_t;

typedef enum {
    CLDT_FRAME_OBSERVATION = 0,
    CLDT_FRAME_COMMAND,
    CLDT_FRAME_ACKNOWLEDGEMENT,
    CLDT_FRAME_CLOCK_SYNC,
    CLDT_FRAME_HEALTH
} cldt_frame_kind_t;

typedef enum {
    CLDT_EVENT_TASK_RELEASE = 0,
    CLDT_EVENT_TASK_START,
    CLDT_EVENT_TASK_FINISH,
    CLDT_EVENT_TASK_BLOCK,
    CLDT_EVENT_QUEUE_ENQUEUE,
    CLDT_EVENT_QUEUE_DEQUEUE,
    CLDT_EVENT_QUEUE_REJECT,
    CLDT_EVENT_POOL_EXHAUSTION,
    CLDT_EVENT_MESSAGE_SEND,
    CLDT_EVENT_MESSAGE_ACK,
    CLDT_EVENT_MESSAGE_EXPIRE,
    CLDT_EVENT_MESSAGE_COALESCE,
    CLDT_EVENT_MESSAGE_DROP,
    CLDT_EVENT_MESSAGE_DUPLICATE,
    CLDT_EVENT_LINK_CHANGE,
    CLDT_EVENT_POWER_SAMPLE,
    CLDT_EVENT_POLICY_APPLY,
    CLDT_EVENT_POLICY_REJECT,
    CLDT_EVENT_POLICY_FALLBACK,
    CLDT_EVENT_HEALTH,
    CLDT_EVENT_COUNT
} cldt_event_kind_t;

typedef enum {
    CLDT_GATE_COLD = 0,
    CLDT_GATE_OBSERVE,
    CLDT_GATE_TRUSTED,
    CLDT_GATE_ABSTAIN
} cldt_gate_state_t;

typedef enum {
    CLDT_MODEL_NAIVE = 0,
    CLDT_MODEL_NETWORK_ONLY,
    CLDT_MODEL_CROSS_LAYER,
    CLDT_MODEL_VARIANT_COUNT
} cldt_model_variant_t;

/*
 * In-memory metadata. It is not a packed wire structure. Encoding and decoding
 * must be performed field by field through cldt_protocol.h.
 *
 * Identity is frame-kind specific. For observations, acknowledgements, health,
 * and trace-bearing frames, node_id/boot_id identify the emitting device. A
 * version 1 command is one global policy datagram for every endpoint admitted
 * to the run: node_id is CLDT_COMMAND_AUTHORITY_NODE_ID and boot_id identifies
 * the host coordinator process that issued it, not a destination. The gateway
 * guards and forwards those identical bytes. Version 1 does not define
 * different authenticated command bytes per endpoint.
 */
typedef struct {
    cldt_frame_kind_t kind;
    cldt_traffic_class_t traffic_class;
    uint16_t flags;
    uint8_t hop_limit;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t transmit_local_us;
    uint64_t deadline_local_us;
} cldt_frame_meta_t;

/*
 * Decoder output borrows payload memory from the input byte buffer. The caller
 * must keep that buffer alive and unchanged while this view is in use.
 */
typedef struct {
    cldt_frame_meta_t meta;
    const uint8_t *payload;
    uint16_t payload_bytes;
    uint32_t crc32c;
    uint8_t authentication_tag[CLDT_AUTH_TAG_BYTES];
} cldt_frame_view_t;

typedef struct {
    cldt_event_kind_t kind;
    /* Every work-item event carries its class; HEALTH may use CLDT_TRAFFIC_COUNT. */
    cldt_traffic_class_t traffic_class;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t local_time_us;
    /*
     * Work-item events repeat the item's release and absolute deadline in the
     * same local monotonic clock domain as local_time_us. Non-work-item events
     * store zero in both fields. This permits stateless aggregate timing while
     * preserving raw timestamps for a separate per-item lifecycle audit.
     */
    uint64_t release_local_us;
    uint64_t deadline_local_us;
    uint32_t task_id;
    int8_t core_id;
    uint16_t queue_depth;
    int16_t link_rssi_dbm;
    uint32_t time_uncertainty_us;
    /* Fixed-size auxiliary bytes; each event kind documents its own encoding. */
    uint8_t detail[CLDT_TRACE_DETAIL_BYTES];
} cldt_trace_record_t;

typedef struct {
    uint32_t release_period_ms[CLDT_POLICY_STREAM_COUNT];
    uint32_t phase_offset_ms[CLDT_POLICY_STREAM_COUNT];
    uint16_t burst_limit[CLDT_POLICY_STREAM_COUNT];
    uint16_t batch_size[CLDT_POLICY_STREAM_COUNT];
    uint32_t token_rate_milli_pps[CLDT_POLICY_STREAM_COUNT];
    cldt_policy_epoch_t epoch;
    uint64_t issued_gateway_us;
    uint32_t ttl_ms;
} cldt_policy_t;

typedef struct {
    cldt_model_variant_t model_variant;
    uint64_t model_revision;
    uint64_t horizon_start_host_us;
    uint64_t horizon_end_host_us;
    uint64_t evaluated_host_us;
    uint64_t newest_observation_host_us;
    uint32_t sample_count;
    uint32_t model_lag_us;
    uint32_t clock_uncertainty_us;
    double relative_p95_error;
    double pdr_error_points;
    double prediction_interval_coverage;
    /* False when required horizon evidence is missing, stale, or unreconciled. */
    bool observation_integrity_valid;
    bool inside_calibrated_region;
} cldt_fidelity_sample_t;

#ifdef __cplusplus
}
#endif

#endif


Offset, enum, identity, payload view, dan ukuran header berasal dari dua header tersebut. `common/include/cldt/cldt_crc32c.h` tetap menjadi dependency CRC yang sudah dibuka pada notebook 1–10.

File berikut adalah source TODO utuh dari repository.

In [ ]:
%%writefile /content/cldt_scratch/cldt_protocol.c
#include "cldt/cldt_protocol.h"

size_t cldt_protocol_encoded_size(size_t payload_bytes)
{
    (void)payload_bytes;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject payload_bytes above CLDT_MAX_PAYLOAD_BYTES before adding it to
     *    CLDT_WIRE_HEADER_BYTES.
     * 2. Perform the addition with an explicit overflow check even though the
     *    current bound is small; this function is the protocol's size gate.
     * 3. Return zero for every unrepresentable input. Callers must treat zero
     *    as a validation failure, never as an empty wire frame.
     * Test with 0, CLDT_MAX_PAYLOAD_BYTES, one byte above the limit, and a
     * SIZE_MAX value. No allocation belongs in this helper.
     */
    return 0U;
}

cldt_status_t cldt_protocol_encode(
    const cldt_frame_meta_t *meta,
    const uint8_t *payload,
    size_t payload_bytes,
    const cldt_authenticator_t *authenticator,
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes)
{
    (void)meta;
    (void)payload;
    (void)payload_bytes;
    (void)authenticator;
    (void)output;
    (void)output_capacity;
    (void)output_bytes;

    /*
     * IMPLEMENTATION TODO:
     * 1. Validate pointer combinations first: a zero-length payload may have a
     *    null payload pointer; a nonzero one may not. Require output_bytes.
     * 2. Ask cldt_protocol_encoded_size() for the exact size and reject a
     *    short output buffer without modifying it or output_bytes.
     * 3. Serialize each header field at the CLDT_WIRE_*_OFFSET declared in
     *    cldt_protocol.h and use network byte order for every multi-byte value.
     *    Write CLDT_WIRE_RESERVED_BYTES as zero. Do not cast output to a packed
     *    C structure: alignment, endianness, and compiler padding would make
     *    the wire contract unstable.
     * 4. Build the canonical integrity sequence by concatenating serialized
     *    header bytes 0-51 with the payload; the CRC and tag slots are omitted,
     *    not included as zero bytes. Calculate CRC-32C over that sequence and
     *    write it at CLDT_WIRE_CRC32C_OFFSET. When authentication is requested,
     *    supply the same sequence as ChaCha20-Poly1305 AAD with zero plaintext
     *    and write the resulting tag. Use bounded caller/stack storage or a
     *    documented scatter/gather helper; no heap allocation belongs here.
     * 5. Write output_bytes only after every validation and authenticator call
     *    succeeds. Add known-answer tests with fixed byte vectors.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_protocol_decode(
    const uint8_t *input,
    size_t input_bytes,
    const cldt_authenticator_t *authenticator,
    bool authentication_required,
    cldt_frame_view_t *output_view)
{
    (void)input;
    (void)input_bytes;
    (void)authenticator;
    (void)authentication_required;
    (void)output_view;

    /*
     * IMPLEMENTATION TODO:
     * 1. Check input and output pointers, then verify that input contains the
     *    fixed header before reading one field. Decode the payload length from
     *    bytes, validate its maximum, and require exact datagram length.
     * 2. Reject wrong magic, unsupported version, invalid enum values, trailing
     *    bytes, nonzero reserved bytes, and malformed flag combinations before
     *    publishing output_view. Read only through the declared offsets.
     * 3. Reconstruct the canonical integrity sequence (header bytes 0-51
     *    concatenated with payload), calculate CRC-32C, and compare it with the
     *    received value before publishing any view.
     * 4. If authentication_required is true, require an authenticator and
     *    verify the received tag over that same sequence as zero-plaintext AAD.
     * 5. Populate output_view only on success. Its payload is a borrowed view,
     *    so never copy a pointer into temporary decoder storage.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_protocol_validate_command(
    const cldt_frame_view_t *frame,
    cldt_run_id_t active_run_id,
    cldt_policy_epoch_t applied_epoch,
    uint64_t now_gateway_us,
    uint32_t time_uncertainty_us)
{
    (void)frame;
    (void)active_run_id;
    (void)applied_epoch;
    (void)now_gateway_us;
    (void)time_uncertainty_us;

    /*
     * IMPLEMENTATION TODO:
     * 1. Accept only CLDT_FRAME_COMMAND after cldt_protocol_decode() has
     *    verified integrity. Never let a health or observation frame enter the
     *    policy path merely because its payload happens to parse.
     * 2. Require exactly CLDT_POLICY_WIRE_BYTES, decode every field through the
     *    declared policy offsets, require frame.meta.run_id to equal the active
     *    run, require payload epoch to equal frame metadata epoch, and require
     *    that epoch to be strictly greater than applied_epoch. Endpoint callers
     *    must supply applied_epoch from a valid durable replay record rather
     *    than resetting it after reboot.
     * 3. Reject a zero or implausibly long TTL. Overflow-check conversion and
     *    addition before comparing issue time plus TTL to now_gateway_us after
     *    expanding the expiry margin by time_uncertainty_us.
     * 4. Return CLDT_ERR_MALFORMED for structural failure, CLDT_ERR_STALE for a
     *    well-formed command outside the accepted freshness window,
     *    CLDT_ERR_WRONG_RUN for another run identity, and CLDT_ERR_EXPIRED for
     *    elapsed TTL. Preserve DUPLICATE and OUT_OF_ORDER for epoch failures.
     * This helper does not persist state or apply a policy. The endpoint caller
     * must durably advance the accepted (run_id, epoch) before publishing the
     * new policy; failure to persist is a rejection and safe fallback. Test
     * duplicate epochs, time-wrap boundaries, and a command that arrives
     * exactly at the uncertainty-expanded expiry limit.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Pekerjaan fungsi, berurutan:

1. `cldt_protocol_encoded_size()` ditutup dahulu. Dengan contract saat ini, payload 0 memerlukan `CLDT_WIRE_HEADER_BYTES`; payload `CLDT_MAX_PAYLOAD_BYTES` memerlukan 328 bytes; nilai di atas batas dan `SIZE_MAX` menghasilkan 0.
2. `cldt_protocol_encode()` memvalidasi seluruh pointer dan capacity sebelum menyentuh output. Multi-byte integer ditulis manual pada setiap `CLDT_WIRE_*_OFFSET` dengan network byte order.
3. Dua reserved byte selalu nol. CRC dihitung atas header byte 0–51 yang langsung diikuti payload; area CRC dan tag tidak ikut.
4. Authenticator bersifat optional pada observation path. Bila callback diberikan, callback menerima canonical integrity sequence yang sama. Implementasi AEAD pada `common/src/cldt_auth.c` tetap menjadi pekerjaan command/safety; Week 3 tidak memuat command key.
5. `cldt_protocol_decode()` menolak datagram yang kurang dari header sebelum membaca payload length, lalu menuntut exact datagram length—trailing byte juga gagal.
6. `output_view` baru dipublikasikan setelah seluruh pemeriksaan berhasil. `payload` tetap borrowed view dari input.
7. `cldt_protocol_validate_command()` tetap diimplementasikan dan diuji sesuai TODO protocol, tetapi tidak dipanggil oleh baseline Week 3.

| Pemeriksaan | Contoh format — ganti dengan hasil test aktual |
|---|---|
| `cldt_protocol_encoded_size(0)` | e.g. 72 |
| `cldt_protocol_encoded_size(CLDT_MAX_PAYLOAD_BYTES)` | e.g. 328 |
| Satu byte di atas `CLDT_MAX_PAYLOAD_BYTES` | e.g. 0 |
| `SIZE_MAX` | e.g. 0 |
| Smallest fixed vector | e.g. PASS — exact bytes cocok |
| Largest fixed vector | e.g. PASS — exact bytes cocok |
| Reserved-byte mutation | e.g. PASS — decode menolak |
| CRC mutation | e.g. PASS — exact status cocok |
| Trailing byte | e.g. PASS — decode menolak |
| Output tidak berubah pada failure | e.g. PASS |

## Step 4: Host Test: Protocol Codec dan Fixed Vectors
### Validasi Test Scaffold tests/test_protocol.c

#### Registration di `tests/CMakeLists.txt`

In [ ]:
%%writefile /content/cldt_scratch/tests_protocol_CMakeLists.txt
function(cldt_add_skeletal_test name source)
    add_executable(${name} ${source})
    target_link_libraries(${name} PRIVATE cldt_common)
    add_test(NAME ${name} COMMAND ${name})
    set_tests_properties(${name} PROPERTIES SKIP_RETURN_CODE 77)
endfunction()

cldt_add_skeletal_test(test_protocol test_protocol.c)
cldt_add_skeletal_test(test_crc32c test_crc32c.c)
cldt_add_skeletal_test(test_auth test_auth.c)
cldt_add_skeletal_test(test_clock_sync test_clock_sync.c)
cldt_add_skeletal_test(test_metrics test_metrics.c)
cldt_add_skeletal_test(test_event_trace test_event_trace.c)
cldt_add_skeletal_test(test_control_profile test_control_profile.c)


`test_protocol` sudah terdaftar dengan `SKIP_RETURN_CODE 77`; tidak ada executable test baru yang diperlukan.

In [ ]:
%%writefile /content/cldt_scratch/test_protocol.c
#include <stdio.h>

#include "cldt/cldt_protocol.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Begin with fixed hexadecimal byte vectors for the smallest and largest
     *    legal frames. Assert every CLDT_WIRE_*_OFFSET, zero reserved bytes,
     *    network byte order, exact size, CRC-32C, and authentication
     *    tag—not only encode/decode round trips, which can hide matching mistakes
     *    on both sides. Mutating either reserved byte must fail decoding.
     * 2. Add rejection cases one mutation at a time: wrong magic, unsupported
     *    version, header/payload length mismatch, truncation at every boundary,
     *    trailing bytes, CRC mutation, authentication mutation, and oversize data.
     * 3. Build one fixed CLDT_POLICY_WIRE_BYTES vector and assert every policy
     *    array/field offset plus equality between payload and metadata epochs.
     *    Test wrong run ID, duplicate epoch, older epoch, stale issue time, zero
     *    TTL, expired TTL, and boundary uncertainty. Verify the exact status,
     *    including CLDT_ERR_STALE and CLDT_ERR_WRONG_RUN, and confirm decoder
     *    output is not partially published.
     *    Gateway integration also rejects a nonzero command-authority node ID or
     *    wrong commissioned authority boot ID without re-encoding the datagram.
     * 4. Keep test vectors in ordinary source data with a short derivation note.
     *    Do not connect Thread or MQTT until these host-only checks are green.
     */
    fprintf(stderr, "SKIP: protocol tests have not been implemented.\n");
    return 77;
}


Urutan case:

1. Buat smallest legal frame sebagai array hexadecimal tetap. Expected byte tidak dihasilkan dengan memanggil encoder yang sedang diuji.
2. Periksa setiap offset dari `CLDT_WIRE_MAGIC_OFFSET` sampai `CLDT_WIRE_AUTH_TAG_OFFSET`, network byte order, reserved bytes, exact size, CRC, dan tag.
3. Buat largest legal frame dengan payload `CLDT_MAX_PAYLOAD_BYTES`.
4. Mutasi satu kondisi per case: magic, version, enum, payload length, setiap truncation boundary, trailing byte, reserved byte, CRC, tag, dan oversize.
5. Gunakan implementasi test untuk callback bertipe `cldt_authenticate_fn` agar callback contract dan fixed tag dapat diuji tanpa mengaktifkan remote actuation. Cryptographic known-answer test milik `tests/test_auth.c` tidak dipalsukan sebagai selesai.
6. Buat satu fixed `CLDT_POLICY_WIRE_BYTES` vector untuk offset/validation command. Ini verification contract, bukan izin mengirim command ke hardware.
7. `return 77` berubah menjadi success hanya setelah seluruh checklist pada komentar source selesai.

| Case | Contoh hasil |
|---|---|
| Smallest encode exact bytes | e.g. PASS |
| Largest encode exact bytes | e.g. PASS |
| Decode smallest/largest | e.g. PASS |
| Truncation pada setiap boundary | e.g. PASS — seluruhnya ditolak |
| Wrong magic/version | e.g. `CLDT_ERR_MALFORMED` / `CLDT_ERR_UNSUPPORTED_VERSION` sesuai contract |
| CRC/tag mutation | e.g. PASS — exact status cocok |
| Wrong run/duplicate/older epoch | e.g. `CLDT_ERR_WRONG_RUN`, `CLDT_ERR_DUPLICATE`, `CLDT_ERR_OUT_OF_ORDER` |
| Exit code `test_protocol` | e.g. 0 |
| `test_auth` pada akhir Week 3 | e.g. SKIPPED — remote command path belum masuk |

## Common — Clock Mapping
### Estimasi Drift dan Uncertainty Timestamp (cldt_clock_sync.h)

Clock mapping menyentuh baseline ketika timestamp endpoint akan dibandingkan dengan gateway/host. Tiga file berikut adalah contract dan TODO repo yang sebenarnya; tidak ada field tambahan dari notebook.

### `common/include/cldt/cldt_clock_sync.h`

In [ ]:
%%writefile /content/cldt_scratch/cldt_clock_sync.h
#ifndef CLDT_CLOCK_SYNC_H
#define CLDT_CLOCK_SYNC_H

#include <stdbool.h>
#include <stdint.h>

#include "cldt/cldt_status.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_CLOCK_SYNC_WINDOW 16U

typedef struct {
    uint64_t request_local_us;
    uint64_t request_gateway_us;
    uint64_t response_gateway_us;
    uint64_t response_local_us;
} cldt_sync_exchange_t;

typedef struct {
    int64_t offset_us;
    int32_t drift_ppm;
    uint32_t uncertainty_us;
    uint32_t accepted_samples;
    uint32_t rejected_samples;
    uint32_t next_slot;
    cldt_sync_exchange_t window[CLDT_CLOCK_SYNC_WINDOW];
    bool valid;
} cldt_clock_sync_t;

void cldt_clock_sync_reset(cldt_clock_sync_t *state);

/* Adds one complete four-timestamp exchange; the implementation owns filtering. */
cldt_status_t cldt_clock_sync_observe(
    cldt_clock_sync_t *state,
    const cldt_sync_exchange_t *exchange);

/* Maps local monotonic time without modifying either device clock. */
cldt_status_t cldt_clock_sync_map_to_gateway(
    const cldt_clock_sync_t *state,
    uint64_t local_time_us,
    uint64_t *gateway_time_us,
    uint32_t *uncertainty_us);

#ifdef __cplusplus
}
#endif

#endif


### `common/src/cldt_clock_sync.c`

In [ ]:
%%writefile /content/cldt_scratch/cldt_clock_sync.c
#include "cldt/cldt_clock_sync.h"

void cldt_clock_sync_reset(cldt_clock_sync_t *state)
{
    (void)state;

    /*
     * IMPLEMENTATION TODO: reject or safely ignore a null state, clear the full
     * exchange window, reset accepted and rejected counts, set offset and drift
     * to neutral values, set uncertainty to an explicit worst-case sentinel,
     * and leave valid false. A reset must occur on boot-ID change, topology
     * reformation, or detected clock anomaly; it must not adjust either clock.
     */
}

cldt_status_t cldt_clock_sync_observe(
    cldt_clock_sync_t *state,
    const cldt_sync_exchange_t *exchange)
{
    (void)state;
    (void)exchange;

    /*
     * IMPLEMENTATION TODO:
     * 1. Validate the four timestamps are ordered consistently with a two-way
     *    exchange and reject arithmetic underflow before calculating delay.
     * 2. Calculate round-trip delay and offset from a complete exchange, then
     *    reject samples whose delay is an outlier relative to the accepted window.
     * 3. Insert accepted samples in the fixed window and estimate offset, drift,
     *    and uncertainty using a documented robust method appropriate for 16
     *    samples; leave valid false until the convergence rule is met.
     * 4. Count rejected samples separately. Never manufacture one-way latency
     *    from an invalid or uncertain mapping.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_clock_sync_map_to_gateway(
    const cldt_clock_sync_t *state,
    uint64_t local_time_us,
    uint64_t *gateway_time_us,
    uint32_t *uncertainty_us)
{
    (void)state;
    (void)local_time_us;
    (void)gateway_time_us;
    (void)uncertainty_us;

    /*
     * IMPLEMENTATION TODO: require a valid state and non-null outputs, apply
     * offset plus drift relative to a documented reference time with checked
     * signed arithmetic, and reject overflow or a negative mapped timestamp.
     * Return the current uncertainty beside the mapped time so each consumer can
     * exclude unsuitable values from one-way deadline analysis. Do not mutate
     * synchronization state from this read-only mapping function.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


### `tests/test_clock_sync.c`

In [ ]:
%%writefile /content/cldt_scratch/test_clock_sync.c
#include <stdio.h>

#include "cldt/cldt_clock_sync.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Use synthetic four-timestamp exchanges with a known offset and drift.
     *    Assert that the estimate remains invalid until the documented minimum
     *    sample rule is met, then maps local time within its reported uncertainty.
     * 2. Add asymmetric-delay and high-round-trip samples. Verify that the chosen
     *    filter either rejects them or expands uncertainty; it must not return a
     *    deceptively precise one-way time.
     * 3. Test timestamp ordering faults, arithmetic near integer boundaries,
     *    boot/reset behavior, drift over a long interval, and output pointers
     *    remaining unchanged on error.
     * 4. The success criterion is honest uncertainty propagation, not merely a
     *    small offset on an ideal synthetic clock.
     */
    fprintf(stderr, "SKIP: clock-sync tests have not been implemented.\n");
    return 77;
}


Panduan implementasinya mengikuti tiga fungsi yang sudah ada:

1. `cldt_clock_sync_reset()` mengosongkan seluruh `window`, counters, offset, drift, dan validity. Nilai sentinel uncertainty dipilih sekali, didokumentasikan, dan diuji; reset dipanggil saat boot identity, partition, atau clock continuity berubah.
2. `cldt_clock_sync_observe()` terlebih dahulu memeriksa ordering dan seluruh pengurangan terhadap underflow. Untuk satu exchange empat timestamp, round-trip delay dikurangi waktu pemrosesan gateway; offset dihitung dari dua arah. Semua arithmetic antara unsigned timestamp dan signed offset diperiksa sebelum conversion.
3. Karena `CLDT_CLOCK_SYNC_WINDOW` bernilai 16, filter, minimum accepted samples, outlier rule, reference time untuk drift, serta convergence rule harus tetap bounded dan tertulis pada source/test. Threshold tidak diisi dari hasil final run.
4. Sampel asymmetric/high-delay boleh ditolak atau memperbesar `uncertainty_us`; ia tidak boleh membuat hasil tampak lebih presisi.
5. `cldt_clock_sync_map_to_gateway()` hanya berhasil bila `valid` benar. Fungsi menyalurkan `uncertainty_us` bersama mapped time dan tidak mengubah state.
6. `test_clock_sync.c` mengganti `return 77` hanya setelah ideal exchange, delayed/asymmetric exchange, ordering faults, near-boundary arithmetic, reset, drift, dan unchanged-output-on-error semuanya mempunyai assertion.
7. Test host dan vector yang sama dijalankan pada endpoint build. Perbedaan integer arithmetic atau overflow harus menjadi failure, bukan toleransi tersembunyi.
8. Sampai test ini lulus dan uncertainty berada di bawah batas yang dibekukan untuk analisis, baseline hanya boleh memakai endpoint-local duration, event order, atau round-trip evidence. One-way latency tidak dilaporkan.

`cldt_sync_exchange_t` saat ini hanya mempunyai in-memory contract. Repo belum mendefinisikan payload clock-sync pada `cldt_protocol.h`, pengirim exchange pada gateway, penerima pada endpoint, atau scheduler exchange. Karena notebook tidak menambah contract baru, synthetic host/device test dapat selesai pada Week 3, tetapi physical one-way clock mapping tetap blocked dan tidak menjadi syarat bagi identity-based lifecycle reconciliation.

| Identifier repo | Contoh evidence setelah implementasi |
|---|---|
| `cldt_clock_sync_t.accepted_samples` | e.g. 12 |
| `cldt_clock_sync_t.rejected_samples` | e.g. 4 |
| `cldt_clock_sync_t.valid` | e.g. true setelah exact convergence rule terpenuhi |
| `cldt_clock_sync_t.offset_us` | e.g. signed result dari synthetic vector |
| `cldt_clock_sync_t.drift_ppm` | e.g. berada dalam tolerance vector yang dibekukan |
| `cldt_clock_sync_t.uncertainty_us` | e.g. mencakup known synthetic error |
| Reset setelah boot/partition change | e.g. PASS — `valid` kembali false |
| Exit code `test_clock_sync` | e.g. 0 setelah semua branch diuji |
| Physical four-timestamp exchange | e.g. BLOCKED — wire/call path belum memiliki contract repo |

## Step 5: Endpoint Thread Transport Adapter
### Integrasi cldt_thread_transport.c dengan Socket OpenThread UDP

### Header contract: `firmware/endpoint/main/thread_transport.h`

In [ ]:
%%writefile /content/cldt_scratch/thread_transport.h
#ifndef CLDT_ENDPOINT_THREAD_TRANSPORT_H
#define CLDT_ENDPOINT_THREAD_TRANSPORT_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_endpoint_command_fn)(
    void *context,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us);

typedef struct {
    uint16_t local_port;
    uint16_t gateway_port;
    uint8_t gateway_ipv6[16];
    cldt_endpoint_command_fn on_command;
    void *callback_context;
} cldt_thread_transport_config_t;

esp_err_t cldt_thread_transport_init(
    const cldt_thread_transport_config_t *config);

/* Caller retains slot ownership until the completion result is returned. */
esp_err_t cldt_thread_transport_send(
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint32_t timeout_ms);

esp_err_t cldt_thread_transport_get_link(
    int8_t *rssi_dbm,
    uint8_t *link_quality,
    uint8_t *thread_role,
    uint32_t *partition_id);

esp_err_t cldt_thread_transport_stop(void);

#ifdef __cplusplus
}
#endif

#endif


### Runtime caller: `firmware/endpoint/main/endpoint_runtime.h`

In [ ]:
%%writefile /content/cldt_scratch/endpoint_runtime.h
#ifndef CLDT_ENDPOINT_RUNTIME_H
#define CLDT_ENDPOINT_RUNTIME_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/event_groups.h"
#include "freertos/task.h"

#include "cldt/cldt_clock_sync.h"
#include "cldt/cldt_event_trace.h"
#include "cldt/cldt_types.h"
#include "deadline_queue.h"
#include "power_probe.h"
#include "workload.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef enum {
    CLDT_ENDPOINT_BOOT = 0,
    CLDT_ENDPOINT_COMMISSIONING,
    CLDT_ENDPOINT_ATTACHED,
    CLDT_ENDPOINT_IDLE,
    CLDT_ENDPOINT_RUNNING,
    CLDT_ENDPOINT_FALLBACK,
    CLDT_ENDPOINT_FAULT
} cldt_endpoint_state_t;

typedef struct {
    cldt_node_id_t node_id;
    cldt_node_role_t role;
    cldt_endpoint_state_t state;
    /* RAM mirrors loaded from an integrity-checked durable replay record. */
    cldt_run_id_t active_run_id;
    cldt_boot_id_t command_authority_boot_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t next_sequence;
    cldt_policy_epoch_t applied_epoch;
    cldt_policy_t safe_policy;
    cldt_policy_t active_policy;
    cldt_clock_sync_t clock_sync;
    cldt_deadline_queue_t deadline_queue;
    cldt_workload_t workload;
    cldt_event_trace_t trace;
    EventGroupHandle_t events;
    TaskHandle_t supervisor_task;
    TaskHandle_t transmitter_task;
    TaskHandle_t trace_task;
    TaskHandle_t power_task;
    /* False forbids remote apply and keeps the compiled safe policy active. */
    bool replay_state_valid;
    bool started;
} cldt_endpoint_runtime_t;

esp_err_t cldt_endpoint_runtime_init(cldt_endpoint_runtime_t *runtime);
esp_err_t cldt_endpoint_runtime_start(cldt_endpoint_runtime_t *runtime);

/* Validates coordinator identity, run, durable epoch, TTL, and local limits. */
esp_err_t cldt_endpoint_runtime_receive_command(
    cldt_endpoint_runtime_t *runtime,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us);

esp_err_t cldt_endpoint_runtime_request_stop(cldt_endpoint_runtime_t *runtime);

#ifdef __cplusplus
}
#endif

#endif


In [ ]:
%%writefile /content/cldt_scratch/thread_transport.c
#include "thread_transport.h"

esp_err_t cldt_thread_transport_init(
    const cldt_thread_transport_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate ports, gateway IPv6 address, callback, and
     * context; start the supported ESP-IDF OpenThread integration; commission or
     * attach using provisioned Thread credentials; wait for an attached state;
     * then bind one project UDP socket. Copy callback configuration into owned
     * state. On any failure, close the socket and unwind OpenThread in reverse
     * order; do not invent a parallel mesh implementation.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_transport_send(
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint32_t timeout_ms)
{
    (void)datagram;
    (void)datagram_bytes;
    (void)timeout_ms;

    /*
     * IMPLEMENTATION TODO: reject null/oversized datagrams and zero or excessive
     * timeout values; transmit one whole UDP datagram; then return a precise send
     * outcome to the transport owner. UDP send success is not delivery success:
     * delivery acknowledgement and retry policy must be handled by the workload
     * state machine so counters distinguish sent, acknowledged, expired, and
     * dropped work. Never hold an OpenThread lock while waiting on a queue.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_transport_get_link(
    int8_t *rssi_dbm,
    uint8_t *link_quality,
    uint8_t *thread_role,
    uint32_t *partition_id)
{
    (void)rssi_dbm;
    (void)link_quality;
    (void)thread_role;
    (void)partition_id;

    /*
     * IMPLEMENTATION TODO: require all output pointers, acquire the OpenThread
     * API lock only long enough to read current RSSI, link quality, role, and
     * partition ID, copy scalar values, then release it before returning. A
     * detached or unavailable role is a valid observable state and should return
     * a clear status or sentinel, not stale data from a previous attachment.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_transport_stop(void)
{
    /*
     * IMPLEMENTATION TODO: reject new sends, unregister receive callbacks, close
     * the project UDP socket, then stop/deinitialize OpenThread as prescribed by
     * ESP-IDF. Clear internal callback state only after no callback can execute.
     * This order prevents a late OpenThread callback from dereferencing endpoint
     * runtime state that the supervisor has already reclaimed.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


Pekerjaan fungsi:

1. `cldt_thread_transport_init()` menyalin semua field `cldt_thread_transport_config_t` ke state milik adapter, memulai integrasi OpenThread yang didukung ESP-IDF, menunggu attached state, lalu bind satu UDP socket project.
2. Callback receive menyalin datagram sebelum callback OpenThread selesai. Pointer milik OpenThread tidak disimpan.
3. `cldt_thread_transport_send()` menolak ukuran 0, ukuran di atas 328 bytes, timeout 0, dan timeout di atas batas yang dibekukan di source.
4. Satu call mengirim satu datagram. Success berarti local UDP send diterima stack, bukan `CLDT_EVENT_MESSAGE_ACK`.
5. `cldt_thread_transport_get_link()` menyalin RSSI, link quality, actual role, dan partition sambil memegang lock sesingkat mungkin.
6. `cldt_thread_transport_stop()` memblokir send baru, melepas callback, menutup socket, lalu deinit OpenThread dalam urutan terbalik.
7. `cldt_endpoint_runtime_t.transmitter_task` tetap memegang slot sampai status send/terminal path dicatat, kemudian memanggil `cldt_deadline_queue_release()`.

| Field/parameter repo | Contoh awal — ganti dengan konfigurasi aktual |
|---|---|
| `cldt_thread_transport_config_t.local_port` | e.g. 1212 |
| `cldt_thread_transport_config_t.gateway_port` | e.g. 1212 |
| `cldt_thread_transport_config_t.gateway_ipv6` | e.g. 16 bytes hasil parse alamat gateway yang benar-benar attached |
| `cldt_thread_transport_config_t.on_command` | e.g. callback tetap menolak command karena remote actuation disabled |
| `cldt_thread_transport_config_t.callback_context` | e.g. pointer ke caller-owned `cldt_endpoint_runtime_t` |
| `timeout_ms` pada send | e.g. 100 |
| Maximum datagram berdasarkan contract | e.g. 328 bytes |
| Hasil send satu frame | e.g. PASS — local send status dan trace tercatat |
| Hasil stop saat partial init | e.g. PASS — tidak ada callback setelah stop |

## Endpoint Build Boundary
### Konfigurasi CMake, Kconfig, dan Entry Point Endpoint

Lima file berikut adalah build dan entry-point contract endpoint yang terhubung ke `thread_transport.c`. Semuanya disalin persis dari repo. Week 3 tidak membuat component atau startup layer kedua.

### `firmware/endpoint/CMakeLists.txt`

In [ ]:
%%writefile /content/cldt_scratch/endpoint_CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(EXTRA_COMPONENT_DIRS "${CMAKE_CURRENT_LIST_DIR}/../../common")

include($ENV{IDF_PATH}/tools/cmake/project.cmake)
project(cldt_endpoint)


### `firmware/endpoint/main/CMakeLists.txt`

In [ ]:
%%writefile /content/cldt_scratch/endpoint_main_CMakeLists.txt
idf_component_register(
    SRCS
        "app_main.c"
        "endpoint_runtime.c"
        "deadline_queue.c"
        "workload.c"
        "thread_transport.c"
        "power_probe.c"
    INCLUDE_DIRS "."
    REQUIRES
        common
        freertos
        nvs_flash
        esp_event
        esp_netif
        esp_timer
        driver
        openthread
)


### `firmware/endpoint/main/Kconfig.projbuild`

In [ ]:
%%writefile /content/cldt_scratch/endpoint_Kconfig.projbuild
menu "CLDT Endpoint"

config CLDT_ENDPOINT_NODE_ID
    int "Provisioning fallback node ID"
    range 1 4294967295
    default 100
    help
        Used only before the endpoint receives a provisioned identity. Record the
        final identity in run evidence; do not use a MAC address or an ad hoc hash.

choice CLDT_ENDPOINT_ROLE
    prompt "Endpoint role"
    default CLDT_ENDPOINT_ROUTER

config CLDT_ENDPOINT_ROUTER
    bool "Router-capable endpoint"
    help
        Enables the project role expected to be router-capable. Actual Thread role
        remains an observed run fact and must not be inferred from this selection.

config CLDT_ENDPOINT_LOW_POWER
    bool "Low-power end-device candidate"
    help
        Selects a candidate end-device configuration for a later admitted study.
        Actual Thread role remains observed, and no version-one energy claim is
        implied by this build choice.

endchoice

config CLDT_ENDPOINT_POOL_SLOTS
    int "Fixed message slots"
    range 8 128
    default 32
    help
        Compile-time queue-pool capacity. Choose from measured peak occupancy and
        preserve a control/critical reservation; never grow it only to hide loss.

config CLDT_ENDPOINT_MAX_TOTAL_RATE_PPS
    int "Compiled maximum aggregate application rate"
    range 1 500
    default 100
    help
        Hard local ceiling across all application streams. A ready manifest may
        request less, but no host policy can raise this value during a run.

config CLDT_ENDPOINT_EVENT_GPIO
    int "Local event input GPIO"
    range -1 30
    default -1
    help
        -1 disables the optional physical event input. Select a pin only after
        checking the exact board revision, boot strapping, pull mode, and debounce
        behavior; XIAO GPIO9 is a boot input and is not a safe generic default.

config CLDT_ENDPOINT_I2C_SDA_GPIO
    int "Optional power-probe I2C SDA GPIO"
    range 0 30
    default 22
    help
        XIAO ESP32-C6 D4/SDA is GPIO22. The version-one power probe is deferred;
        recheck the exact board and record wiring before a future energy pilot.

config CLDT_ENDPOINT_I2C_SCL_GPIO
    int "Optional power-probe I2C SCL GPIO"
    range 0 30
    default 23
    help
        XIAO ESP32-C6 D5/SCL is GPIO23. Do not assume another C6 board variant
        shares this mapping.

endmenu


### `firmware/endpoint/sdkconfig.defaults`

In [ ]:
%%writefile /content/cldt_scratch/endpoint_sdkconfig.defaults
CONFIG_FREERTOS_HZ=1000
CONFIG_FREERTOS_USE_TRACE_FACILITY=y
CONFIG_FREERTOS_GENERATE_RUN_TIME_STATS=y
CONFIG_ESP_TASK_WDT_EN=y
CONFIG_OPENTHREAD_ENABLED=y
CONFIG_MBEDTLS_CHACHAPOLY_C=y
CONFIG_MBEDTLS_CHACHA20_C=y
CONFIG_MBEDTLS_POLY1305_C=y


### `firmware/endpoint/main/app_main.c`

In [ ]:
%%writefile /content/cldt_scratch/endpoint_app_main.c
#include "esp_log.h"

static const char *TAG = "cldt_endpoint";

void app_main(void)
{
    ESP_LOGW(TAG,
             "Research scaffold only: endpoint runtime, deadline queue, "
             "workload, Thread transport, and power probe are not implemented.");

    /*
     * IMPLEMENTATION ORDER:
     * 1. Keep networking disabled while proving local software-timer, ISR,
     *    queue ownership, fixed-pool exhaustion, expiry, and counter tests.
     * 2. Add Thread attachment only after those tests produce reconciled traces.
     * 3. Add command handling only after protocol known-answer and replay tests.
     * 4. Enable the optional power probe last and document its overhead.
     * This entry point should remain small: construct runtime, call init/start,
     * and hand lifecycle ownership to the supervisor. It is not a demo script.
     */
}


Pekerjaan build/integrasinya:

1. Root endpoint project tetap mengambil `common` melalui `EXTRA_COMPONENT_DIRS`; checkout ESP-IDF dan target `esp32c6` berasal dari toolchain yang dibekukan.
2. `firmware/endpoint/main/CMakeLists.txt` sudah memasukkan `thread_transport.c`. Tidak ada source baru yang perlu ditambahkan untuk menghubungkan transport. Jika include/link OpenThread gagal, diagnosis diarahkan ke requirement dan ESP-IDF revision, bukan membuat adapter duplikat.
3. `Kconfig.projbuild` hanya menyediakan identifier yang tampak pada snippet. Ia belum mempunyai project UDP port atau gateway IPv6 setting; nilai tersebut tidak boleh diselundupkan melalui label notebook.
4. `sdkconfig.defaults` mengaktifkan OpenThread dan crypto backend. Actual `sdkconfig` hasil build, target, serta binary digest disimpan sebagai evidence bersama source revision.
5. `app_main.c` tetap hanya membuat caller-owned runtime, memanggil `cldt_endpoint_runtime_init()` dan `cldt_endpoint_runtime_start()`, lalu menyerahkan lifecycle kepada supervisor. Queue, socket, timer, broker, atau test loop tidak diletakkan di entry point.
6. Baris scaffold log diganti hanya saat pesan itu tidak lagi benar. Log startup baru menyatakan state yang benar-benar dicapai, bukan “ready” sebelum attach/runtime prerequisite selesai.
7. Build local-only Week 2 dan build Thread Week 3 dipisahkan oleh exact `sdkconfig`/binary identity. Perbedaan konfigurasi dicatat; source branch tidak dibuat diam-diam.

| Build evidence | Contoh format |
|---|---|
| ESP-IDF commit | e.g. 40-character commit dari checkout yang sama |
| Target | e.g. esp32c6 |
| SHA-256 actual `sdkconfig` | e.g. 64 hexadecimal characters |
| SHA-256 endpoint binary | e.g. 64 hexadecimal characters |
| `thread_transport.c` pada component source list | e.g. PASS |
| local-only regression | e.g. PASS |
| Thread build/flash | e.g. PASS dengan raw build dan boot log |

## Endpoint Runtime Integration
### Penyambungan Transmitter Task dengan Protocol Codec

Source ini sudah menjadi tempat local-only runtime pada Week 2 dan menjadi caller `thread_transport.c` pada Week 3. Snippet berikut adalah isi persis `firmware/endpoint/main/endpoint_runtime.c` pada commit acuan; TODO command tetap terlihat karena bagian itu belum termasuk jalur observation-only.

In [ ]:
%%writefile /content/cldt_scratch/endpoint_runtime.c
#include "endpoint_runtime.h"

esp_err_t cldt_endpoint_runtime_init(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: reject a null runtime, clear caller-owned state, load
     * immutable board identity and role, generate a boot ID that changes after a
     * reset, and load the integrity-checked durable replay record containing the
     * enrolled run, coordinator boot identity, and highest accepted epoch. A
     * missing or corrupt record leaves
     * replay_state_valid false: retain the compiled safe policy and require an
     * explicitly commissioned new unique run before remote apply. Create every
     * steady-state queue, trace buffer, event group, and task storage statically
     * and leave the state at BOOT. No radio attach, workload release, or dynamic
     * allocation is permitted here. Fail before changing externally visible
     * state on any error.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_start(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: require successful initialization, then start the
     * supervisor task first. It owns transitions through commissioning, attach,
     * idle, running, fallback, and fault. Start transport, workload, trace, and
     * optional power tasks only after the supervisor reports their prerequisites;
     * if any task creation fails, notify supervisor to unwind already started
     * components. Do not start release timers merely because Thread attached.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_receive_command(
    cldt_endpoint_runtime_t *runtime,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us)
{
    (void)runtime;
    (void)datagram;
    (void)datagram_bytes;
    (void)received_local_us;

    /*
     * IMPLEMENTATION TODO:
     * 1. Copy or retain the datagram only for the duration required by the
     *    decoder; reject oversized input before queueing work.
     * 2. Decode and authenticate it; require coordinator authority node ID 0 and
     *    the coordinator boot/session identity commissioned for this run, the
     *    enrolled run ID, a valid durable replay state, a strictly newer epoch,
     *    a live TTL, and endpoint-local limits. Map received_local_us into the
     *    gateway monotonic domain through the validated clock-sync state and
     *    reject excessive uncertainty; never compare unrelated local clocks.
     *    The command boot ID identifies the coordinator process, not this
     *    endpoint and not replay state. A host decision is not local authorization.
     * 3. Atomically persist the new (run_id, coordinator_boot_id, highest_epoch)
     *    before publishing
     *    one immutable policy snapshot at a workload release boundary. If the
     *    durable write fails, reject and retain the safe policy. A duplicate
     *    accepted epoch must be acknowledged as duplicate, never applied twice.
     * 4. Emit a trace record and an acknowledgement for every accept or reject
     *    reason. On missing/corrupt replay state or any ambiguity, preserve the
     *    safe policy, enter FALLBACK, and require a newly commissioned run.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_request_stop(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: request a supervisor-owned stop, block new workload
     * releases, let producer and transport finish or explicitly expire queued
     * work, request final counters, and reconcile before changing state to IDLE.
     * A stopped endpoint must retain its safe policy and remain able to report
     * health. Do not delete a task from an arbitrary caller or discard evidence
     * merely to make shutdown appear fast.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


Bagian Week 3 dikerjakan tanpa menghapus perilaku local accounting yang sudah lulus:

1. `cldt_endpoint_runtime_init()` tetap tidak melakukan attach atau release. Static storage, identity, safe policy, queue, workload, dan trace yang diselesaikan pada Week 2 menjadi prasyarat; transport configuration hanya disiapkan dalam ownership runtime.
2. `cldt_endpoint_runtime_start()` menjaga supervisor sebagai pemilik transisi state. Transport dimulai setelah prerequisite commissioning tersedia, tetapi workload baru release setelah state running dan run context baseline telah diterima.
3. `transmitter_task` yang sudah tercantum pada `cldt_endpoint_runtime_t` menjadi satu-satunya owner dequeue–encode–send–terminal. Ia memanggil `cldt_protocol_encoded_size()`, `cldt_protocol_encode()`, dan `cldt_thread_transport_send()` sebelum slot dilepas melalui `cldt_deadline_queue_release()`.
4. Local UDP send success dicatat sebagai send outcome, bukan acknowledgement end to end. ACK/expiry/drop memakai event dan counter yang sudah dibekukan pada Week 2.
5. Receive callback boleh meneruskan observation acknowledgement yang termasuk baseline path, tetapi `cldt_endpoint_runtime_receive_command()` tetap fail-closed dan remote actuation tetap disabled. TODO authentication, durable epoch, TTL, dan apply policy menjadi pekerjaan safety phase, bukan Week 3.
6. `cldt_endpoint_runtime_request_stop()` menghentikan release baru, menunggu atau menandai terminal untuk item yang tersisa, mengambil final counters, menghentikan transport, lalu mengembalikan state ke idle melalui supervisor.
7. Uji regresi local-only dijalankan lagi setelah transport terhubung. Branch radio-off harus tetap menghasilkan lifecycle yang sama tanpa meminta `thread_transport.c`.

| Identifier repo | Contoh evidence setelah implementasi |
|---|---|
| `cldt_endpoint_runtime_t.state` | e.g. BOOT → COMMISSIONING → ATTACHED → IDLE → RUNNING |
| `cldt_endpoint_runtime_t.transmitter_task` | e.g. static task handle non-null setelah start |
| `cldt_endpoint_runtime_t.active_run_id` | e.g. sama dengan ready baseline manifest |
| `cldt_endpoint_runtime_t.next_sequence` | e.g. monotonik dalam satu boot |
| Send trace sebelum slot release | e.g. PASS |
| Queue drain pada stop | e.g. semua item terminal atau explicit unresolved |
| `cldt_endpoint_runtime_receive_command()` | e.g. rejected / not enabled pada baseline |
| local-only regression | e.g. PASS tanpa OpenThread init |

## Step 6: Gateway Provisioning Prerequisite
### Manajemen Kredensial Wi-Fi dan Parameter Thread Tanpa Hardcoded Secrets

### Header contract: `firmware/gateway/main/gateway_provisioning.h`

In [ ]:
%%writefile /content/cldt_scratch/gateway_provisioning.h
#ifndef CLDT_GATEWAY_PROVISIONING_H
#define CLDT_GATEWAY_PROVISIONING_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_WIFI_SSID_MAX_BYTES 32U
#define CLDT_WIFI_PASSPHRASE_MAX_BYTES 64U
#define CLDT_BROKER_URI_MAX_BYTES 128U
#define CLDT_COMMAND_KEY_BYTES 32U

typedef struct {
    char wifi_ssid[CLDT_WIFI_SSID_MAX_BYTES + 1U];
    char wifi_passphrase[CLDT_WIFI_PASSPHRASE_MAX_BYTES + 1U];
    char broker_uri[CLDT_BROKER_URI_MAX_BYTES];
    uint8_t command_key[CLDT_COMMAND_KEY_BYTES];
    uint32_t node_id;
} cldt_gateway_credentials_t;

/* Loads validated credentials from encrypted/protected NVS where available. */
esp_err_t cldt_gateway_provisioning_load(
    cldt_gateway_credentials_t *output,
    bool *is_provisioned);

/*
 * Runs a temporary authenticated BLE GATT service. The implementation must
 * require physical presence, bound every characteristic, and stop advertising
 * before a measured run begins.
 */
esp_err_t cldt_gateway_provisioning_start_ble(void);

esp_err_t cldt_gateway_provisioning_stop_ble(void);

/* Erasure must require a deliberate local action; never expose it over MQTT. */
esp_err_t cldt_gateway_provisioning_erase(void);

#ifdef __cplusplus
}
#endif

#endif


In [ ]:
%%writefile /content/cldt_scratch/gateway_provisioning.c
#include "gateway_provisioning.h"

esp_err_t cldt_gateway_provisioning_load(
    cldt_gateway_credentials_t *output,
    bool *is_provisioned)
{
    (void)output;
    (void)is_provisioned;

    /*
     * IMPLEMENTATION TODO: initialize the approved NVS namespace, read only
     * project-owned keys into bounded temporary buffers, validate lengths and
     * NUL termination before copying to output, and report provisioned false for
     * incomplete data. Never print SSID, passphrase, broker URI credentials, or
     * command key. Zero temporary secret buffers on every error and success path.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_provisioning_start_ble(void)
{
    /*
     * IMPLEMENTATION TODO: expose only the characteristics needed to provision
     * gateway identity, Wi-Fi credentials, broker endpoint, and command trust
     * material; require explicit local physical presence before advertising; and
     * use authenticated pairing or a documented secure enrollment procedure. Bound
     * every write length, reject reads of secrets, and persist only after complete
     * validation. BLE is provisioning-only and never carries measured telemetry.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_provisioning_stop_ble(void)
{
    /*
     * IMPLEMENTATION TODO: stop advertising and GATT service, disconnect active
     * clients, zero temporary pairing and credential buffers, release NimBLE
     * resources as required by ESP-IDF, and emit only a non-sensitive state
     * transition. Require this operation before a measured run so BLE coexistence
     * cannot become an undocumented 2.4 GHz treatment variable.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_provisioning_erase(void)
{
    /*
     * IMPLEMENTATION TODO: sample a designated physical button with debounce and
     * a long-press confirmation window, visibly indicate pending erase without
     * exposing secrets, delete only the project's NVS namespace, zero in-memory
     * copies, and reboot into unprovisioned state. MQTT, HTTP, and BLE must never
     * be able to invoke this operation remotely.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


Backhaul menerima `cldt_gateway_credentials_t`; karena itu credential path bukan pekerjaan tambahan acak. Ia adalah prerequisite dari source yang sudah ada.

1. `cldt_gateway_provisioning_load()` memulai NVS namespace project, membaca bounded value, memvalidasi terminator/panjang, dan membersihkan temporary buffer pada setiap exit.
2. SSID, passphrase, URI dengan credential, dan `command_key` tidak pernah ditulis ke log atau notebook.
3. Observation-only run mempunyai `remote_actuation` false. Current struct tetap memuat `command_key`; source harus memutuskan secara eksplisit apakah provisioning observation-only boleh lengkap tanpa key. Notebook tidak mengisi key dummy.
4. `cldt_gateway_provisioning_start_ble()` hanya berjalan setelah physical presence, dengan authenticated enrollment dan bounded characteristic.
5. `cldt_gateway_provisioning_stop_ble()` selesai sebelum baseline diukur supaya BLE tidak menjadi traffic 2.4 GHz tersembunyi.
6. `cldt_gateway_provisioning_erase()` hanya dari deliberate local action; MQTT/HTTP/BLE tidak boleh memanggilnya.

| Field `cldt_gateway_credentials_t` | Contoh bentuk pencatatan aman |
|---|---|
| `wifi_ssid` | e.g. tersimpan di NVS; nilai tidak disalin ke notebook |
| `wifi_passphrase` | e.g. tersimpan di NVS; nilai tidak disalin ke notebook |
| `broker_uri` | e.g. mqtt://192.168.1.20:1883 pada private LAN |
| `command_key` | e.g. not provisioned untuk observation-only; jangan isi zero key sebagai key valid |
| `node_id` | e.g. 1; harus cocok dengan identity gateway yang diprovision |
| Hasil `cldt_gateway_provisioning_load()` | e.g. `ESP_OK`, `is_provisioned == true` |
| BLE berhenti sebelum measurement | e.g. PASS |
| Secret muncul di log | e.g. 0 occurrence |

## Step 7: Gateway Thread Ingress
### Penerimaan Datagram OpenThread dan Snapshot Topologi (thread_bridge.h)

### Header contracts

#### `firmware/gateway/main/thread_bridge.h`

In [ ]:
%%writefile /content/cldt_scratch/thread_bridge.h
#ifndef CLDT_GATEWAY_THREAD_BRIDGE_H
#define CLDT_GATEWAY_THREAD_BRIDGE_H

#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_thread_frame_fn)(
    void *context,
    const uint8_t *datagram,
    size_t datagram_bytes,
    const uint8_t source_ipv6[16],
    int8_t rssi_dbm,
    uint64_t received_local_us);

typedef struct {
    cldt_thread_frame_fn on_frame;
    void *callback_context;
    uint16_t listen_port;
} cldt_thread_bridge_config_t;

/* Attaches the project UDP adapter after the upstream border router is ready. */
esp_err_t cldt_thread_bridge_init(const cldt_thread_bridge_config_t *config);

esp_err_t cldt_thread_bridge_send(
    const uint8_t destination_ipv6[16],
    const uint8_t *datagram,
    size_t datagram_bytes);

/* Captures current role/partition/neighbor state into caller-owned output. */
esp_err_t cldt_thread_bridge_snapshot(
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/gateway/main/thread_diagnostic.h`

In [ ]:
%%writefile /content/cldt_scratch/thread_diagnostic.h
#ifndef CLDT_GATEWAY_THREAD_DIAGNOSTIC_H
#define CLDT_GATEWAY_THREAD_DIAGNOSTIC_H

#include <stdint.h>
#include <stdbool.h>
#include "cldt/cldt_status.h"

typedef struct {
    uint32_t tx_total;
    uint32_t tx_retry;
    uint32_t tx_err_cca;
    uint32_t tx_direct_max_retry_expiry;
    uint32_t rx_total;
    uint32_t rx_err_fcs;
    uint32_t rx_duplicated;
} cldt_mac_snapshot_t;

typedef struct {
    uint16_t parent_rloc16;
    uint8_t parent_link_quality_in;
    uint8_t parent_link_quality_out;
    uint32_t partition_id;
    uint8_t device_role;
    bool parent_changed;
    bool role_changed;
    bool partition_changed;
} cldt_thread_state_t;

typedef struct {
    cldt_mac_snapshot_t mac_delta;
    cldt_thread_state_t thread_state;
    uint64_t timestamp_us;
    uint32_t observation_count;
} cldt_cross_layer_observation_t;

cldt_status_t cldt_thread_diagnostic_init(void);
cldt_status_t cldt_thread_diagnostic_poll(cldt_cross_layer_observation_t *output);

#endif // CLDT_GATEWAY_THREAD_DIAGNOSTIC_H


#### `firmware/gateway/main/thread_bridge.c`

In [ ]:
%%writefile /content/cldt_scratch/thread_bridge.c
#include "thread_bridge.h"

esp_err_t cldt_thread_bridge_init(const cldt_thread_bridge_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate callback, context, and listen port; confirm
     * the upstream border router and RCP are initialized and attached; then bind
     * one project UDP socket. The receive callback must copy or enqueue a bounded
     * datagram plus source metadata and return promptly. Do not retain pointers
     * owned by OpenThread, and make RCP/Thread detach visible to gateway runtime.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_bridge_send(
    const uint8_t destination_ipv6[16],
    const uint8_t *datagram,
    size_t datagram_bytes)
{
    (void)destination_ipv6;
    (void)datagram;
    (void)datagram_bytes;

    /*
     * IMPLEMENTATION TODO: require a non-null 16-byte destination and one bounded
     * datagram, verify protocol size before taking any OpenThread API lock, send
     * exactly one datagram, and return a precise local send status. This function
     * consumes no caller buffer ownership; caller may reuse its memory after return
     * only if the upstream API has copied it. Document that behavior explicitly.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_bridge_snapshot(
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes)
{
    (void)output;
    (void)output_capacity;
    (void)output_bytes;

    /*
     * IMPLEMENTATION TODO: require output/output_bytes, acquire the OpenThread
     * lock briefly, serialize only the selected current role, partition, parent,
     * and neighbor/link fields into caller-owned bytes, then release the lock.
     * Bound every list and output length; a partial snapshot must be marked as
     * partial rather than presented as complete topology evidence.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


#### `firmware/gateway/main/thread_diagnostic.c`

In [ ]:
%%writefile /content/cldt_scratch/thread_diagnostic.c
#include "thread_diagnostic.h"
#include "esp_openthread.h"
#include <openthread/instance.h>
#include <openthread/link.h>
#include <openthread/thread.h>
#include <stddef.h>

static cldt_mac_snapshot_t s_last_mac_snapshot = {0};
static uint16_t s_last_parent_rloc16 = 0xFFFF;
static uint8_t s_last_device_role = 0xFF;
static uint32_t s_last_partition_id = 0xFFFFFFFF;
static uint32_t s_observation_count = 0;

cldt_status_t cldt_thread_diagnostic_init(void)
{
    otInstance *instance = esp_openthread_get_instance();
    if (instance == NULL) {
        return CLDT_ERR_NOT_READY;
    }
    
    // TODO: ESP-IDF v5.2+ for stable OpenThread support
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_thread_diagnostic_poll(cldt_cross_layer_observation_t *output)
{
    // TODO: Thread-safety: wrap in OpenThread task lock or dedicated mutex
    // TODO: otLinkGetCounters(instance) returns const otMacCounters*: fields mTxTotal, mTxRetry, mTxErrCca, mTxDirectMaxRetryExpiry, mRxTotal, mRxErrFcs, mRxDuplicated
    // TODO: Delta calculation: current_value - s_last_snapshot_value for each field, then update snapshot
    // TODO: otThreadGetParentInfo(instance, &parent_info): otRouterInfo has mRloc16, mLinkQualityIn (0-3), mLinkQualityOut (0-3), mExtAddress
    // TODO: Parent change detection: compare parent_info.mRloc16 != s_last_parent_rloc16
    // TODO: otThreadGetLeaderData(instance, &leader_data): otLeaderData has mPartitionId, mWeighting, mLeaderRouterId
    // TODO: Partition change: compare leader_data.mPartitionId != s_last_partition_id
    // TODO: otThreadGetDeviceRole(instance): returns OT_DEVICE_ROLE_DISABLED/_DETACHED/_CHILD/_ROUTER/_LEADER
    // TODO: Role change: compare current_role != s_last_device_role
    // TODO: Call these APIs on the S3 OpenThread host. Verify which counters the
    // pinned host/RCP configuration populates before using them as evidence.
    // TODO: Polling interval: configurable, default 1 second, use esp_timer for periodic callback
    
    return CLDT_ERR_NOT_IMPLEMENTED;
}


#### Current component list

In [ ]:
%%writefile /content/cldt_scratch/gateway_main_CMakeLists.txt
idf_component_register(
    SRCS
        "app_main.c"
        "gateway_runtime.c"
        "gateway_provisioning.c"
        "thread_bridge.c"
        "backhaul.c"
        "policy_guard.c"
    INCLUDE_DIRS "."
    REQUIRES
        common
        freertos
        nvs_flash
        esp_event
        esp_netif
        esp_wifi
        esp_timer
        bt
        mqtt
        esp_http_server
        openthread
)


Urutan kerja:

1. Tambahkan `thread_diagnostic.c` ke `SRCS`; pada checkout sekarang file itu belum dikompilasi oleh component.
2. `cldt_thread_bridge_init()` hanya berhasil setelah upstream border router/RCP ready dan attached. `on_frame`, `callback_context`, serta `listen_port` divalidasi dan disalin.
3. Receive callback membuat bounded copy berisi datagram dan source metadata, lalu enqueue ke `cldt_gateway_runtime_t.thread_rx_queue`. Callback tidak decode berat, publish MQTT, atau menunggu reconnect.
4. Jika queue penuh, datagram tidak overwrite record lama. Drop dan high-water menjadi health evidence.
5. `cldt_thread_bridge_snapshot()` menulis hanya sebanyak `output_capacity`; incomplete topology ditandai partial, bukan disebut complete.
6. `cldt_thread_diagnostic_init()` menyimpan baseline counter pertama.
7. `cldt_thread_diagnostic_poll()` membaca `otMacCounters`, parent, partition, role, lalu menghasilkan delta dan change flag ke `cldt_cross_layer_observation_t`.
8. Semua OpenThread API dipanggil di bawah lock yang benar, tetapi serialization/enqueue terjadi setelah lock dilepas.

| Field/config repo | Contoh awal — ganti dari build/pilot |
|---|---|
| `cldt_thread_bridge_config_t.listen_port` | e.g. 1212 |
| Kconfig `CLDT_GATEWAY_RX_QUEUE_LENGTH` | e.g. 32, lalu buktikan high-water |
| Kconfig `CLDT_GATEWAY_TRACE_QUEUE_LENGTH` | e.g. 128, lalu buktikan high-water |
| `cldt_mac_snapshot_t.tx_total` delta | e.g. 120 |
| `cldt_mac_snapshot_t.tx_retry` delta | e.g. 4 |
| `cldt_thread_state_t.parent_rloc16` | e.g. 0x1400 atau not applicable sesuai actual role |
| `cldt_thread_state_t.partition_id` | e.g. 123456789 |
| `cldt_thread_state_t.device_role` | e.g. nilai `otDeviceRole` aktual |
| `cldt_cross_layer_observation_t.observation_count` | e.g. bertambah satu per poll sukses |
| Polling cadence dari komentar source | e.g. 1 s setelah overhead diukur |

## Gateway Edge Safety Guard
### Kontrak Policy Guard pada Mode Disarmed (policy_guard.h)

`cldt_gateway_runtime_t` selalu memuat `cldt_policy_guard_t`, dan `gateway_runtime.c` meminta guard diinisialisasi sebelum task hidup. Karena file ini sudah terdaftar pada component, observation-only baseline tetap memerlukan subset init/run/end yang aman; ia tidak memerlukan penerimaan proposal.

### `firmware/gateway/main/policy_guard.h`

In [ ]:
%%writefile /content/cldt_scratch/policy_guard.h
#ifndef CLDT_GATEWAY_POLICY_GUARD_H
#define CLDT_GATEWAY_POLICY_GUARD_H

#include <stdbool.h>
#include <stdint.h>

#include "cldt/cldt_control_profile.h"
#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    /*
     * Effective per-run ceilings are derived from the resolved control profile
     * and capped by compiled safety maxima. A manifest cannot raise them.
     */
    uint32_t maximum_total_rate_pps;
    uint32_t minimum_critical_period_ms;
    uint16_t maximum_bulk_burst_packets;
    uint32_t maximum_policy_ttl_ms;
} cldt_edge_limits_t;

typedef struct {
    /* Only this object owns the mutable edge policy snapshot and applied epoch. */
    cldt_run_id_t active_run_id;
    cldt_boot_id_t command_authority_boot_id;
    cldt_policy_epoch_t applied_epoch;
    /*
     * Archived with every decision so a gateway trace can identify the exact
     * resolved profile even if a human-readable profile name is later reused.
     */
    uint8_t control_profile_digest[CLDT_CONTROL_PROFILE_DIGEST_BYTES];
    cldt_policy_t safe_fallback;
    cldt_policy_t active_policy;
    cldt_edge_limits_t limits;
    bool remote_actuation_enabled;
} cldt_policy_guard_t;

cldt_status_t cldt_policy_guard_init(
    cldt_policy_guard_t *guard,
    const cldt_control_profile_t *control_profile,
    const cldt_edge_limits_t *limits,
    const cldt_policy_t *safe_fallback);

/*
 * Binds the guard to one nonzero run already reserved in the global run ledger
 * and resets it to the safe policy. An actuated run is additionally bound to
 * the command-key identity. A gateway boot never resumes forwarding an old run.
 * Remote actuation remains fail-closed unless both the build maturity switch
 * and the frozen ready manifest authorize it.
 */
cldt_status_t cldt_policy_guard_begin_run(
    cldt_policy_guard_t *guard,
    cldt_run_id_t run_id,
    cldt_boot_id_t command_authority_boot_id,
    bool remote_actuation_requested);

/*
 * Validates host output independently; it does not trust the host gate state.
 * Caller supplies local time, uncertainty, and health observed at acceptance.
 * On failure, the function must leave the active policy and epoch unchanged.
 */
cldt_status_t cldt_policy_guard_accept(
    cldt_policy_guard_t *guard,
    cldt_run_id_t proposal_run_id,
    cldt_boot_id_t proposal_authority_boot_id,
    const cldt_policy_t *proposal,
    uint64_t now_gateway_us,
    uint32_t clock_uncertainty_us,
    bool local_health_ok);

/*
 * Immediately selects the compiled safe policy and records the reason. This
 * must be usable while host, broker, or model connectivity is absent.
 */
cldt_status_t cldt_policy_guard_fallback(
    cldt_policy_guard_t *guard,
    cldt_status_t reason);

/* Disarms remote control, restores fallback, and clears run/epoch after trace. */
cldt_status_t cldt_policy_guard_end_run(
    cldt_policy_guard_t *guard,
    cldt_status_t terminal_reason);

#ifdef __cplusplus
}
#endif

#endif


### `firmware/gateway/main/policy_guard.c`

In [ ]:
%%writefile /content/cldt_scratch/policy_guard.c
#include "policy_guard.h"

cldt_status_t cldt_policy_guard_init(
    cldt_policy_guard_t *guard,
    const cldt_control_profile_t *control_profile,
    const cldt_edge_limits_t *limits,
    const cldt_policy_t *safe_fallback)
{
    (void)guard;
    (void)control_profile;
    (void)limits;
    (void)safe_fallback;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject null arguments and validate the resolved control profile before
     *    accepting any edge limits. The caller must calculate limits by taking
     *    the stricter value of profile settings and build-time safety maxima.
     * 2. Copy the profile digest, effective limits, and safe fallback into
     *    guard-owned storage. Validate fallback with the same arithmetic used
     *    for host proposals; a fallback may never violate the effective limits.
     * 3. Set active policy to the safe fallback, clear active run, command
     *    authority boot ID, and applied epoch, and begin with
     *    remote_actuation_enabled false. A failed
     *    initialization must leave no policy that could be mistaken for an
     *    accepted remote command.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_begin_run(
    cldt_policy_guard_t *guard,
    cldt_run_id_t run_id,
    cldt_boot_id_t command_authority_boot_id,
    bool remote_actuation_requested)
{
    (void)guard;
    (void)run_id;
    (void)command_authority_boot_id;
    (void)remote_actuation_requested;

    /*
     * IMPLEMENTATION TODO: require an initialized guard, nonzero run ID, no
     * active run, a nonzero command_authority_boot_id, and admission evidence
     * that the host reserved this ID in the durable global run ledger. If remote
     * actuation is requested, also require its binding to the active command-key
     * identity. A fresh gateway boot must refuse to resume forwarding an old
     * run. Only after that uniqueness boundary is proven may the guard restore
     * fallback, reset applied_epoch, and bind run_id plus command authority.
     * Enable remote acceptance only when requested by the frozen ready manifest
     * and CONFIG_CLDT_GATEWAY_REMOTE_ACTUATION is enabled; otherwise keep the run
     * shadow-only. Trace the resulting mode.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_accept(
    cldt_policy_guard_t *guard,
    cldt_run_id_t proposal_run_id,
    cldt_boot_id_t proposal_authority_boot_id,
    const cldt_policy_t *proposal,
    uint64_t now_gateway_us,
    uint32_t clock_uncertainty_us,
    bool local_health_ok)
{
    (void)guard;
    (void)proposal_run_id;
    (void)proposal_authority_boot_id;
    (void)proposal;
    (void)now_gateway_us;
    (void)clock_uncertainty_us;
    (void)local_health_ok;

    /*
     * IMPLEMENTATION TODO: hold the short policy critical section only while
     * checking remote_actuation_enabled, local health, exact proposal_run_id and
     * proposal_authority_boot_id equality with the commissioned run authority,
     * strictly increasing epoch, finite TTL, clock
     * uncertainty, critical-period protection, bulk burst limit, and total rate.
     * Validate the full proposal before swapping it. On any failure leave the
     * active policy and epoch unchanged, return a reason code, and ensure the
     * caller emits a rejection trace/acknowledgement outside the lock. On
     * success the caller forwards the retained authenticated datagram byte for
     * byte; it must not re-encode different bytes under the accepted nonce.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_end_run(
    cldt_policy_guard_t *guard,
    cldt_status_t terminal_reason)
{
    (void)guard;
    (void)terminal_reason;

    /*
     * IMPLEMENTATION TODO: disarm remote acceptance first, atomically restore
     * the safe policy, emit the final policy/epoch/run/authority record, then
     * clear the active run, command authority, and applied epoch. Never clear
     * identity before the terminal trace is durable enough for the gateway's
     * evidence path.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_fallback(
    cldt_policy_guard_t *guard,
    cldt_status_t reason)
{
    (void)guard;
    (void)reason;

    /*
     * IMPLEMENTATION TODO: atomically replace active_policy with the compiled
     * safe snapshot and record the supplied failure reason and current accepted
     * epoch in a traceable event. Retain active_run_id and applied_epoch so a
     * later requalified command must still advance strictly. Stop forwarding
     * new proposals. Endpoints return to their own compiled safe policy when
     * the last accepted finite command
     * expires; version one must not synthesize a second command under the host's
     * nonce space. Fallback must succeed without host, broker, or model access.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Pembagian pekerjaan Week 3 dan pekerjaan safety phase tidak dicampur:

1. `cldt_policy_guard_init()` memvalidasi resolved profile, effective `cldt_edge_limits_t`, serta `safe_fallback`; menyalinnya ke guard-owned storage; mengosongkan run/authority/epoch; lalu menetapkan `remote_actuation_enabled` false.
2. Effective limit selalu merupakan nilai yang lebih ketat antara control profile dan Kconfig gateway. Arithmetic fallback diuji terhadap limit yang sama sebelum active policy diisi.
3. `cldt_policy_guard_begin_run()` menerima run identity yang sudah direservasi launcher. Untuk baseline, `remote_actuation_requested` false dan `CONFIG_CLDT_GATEWAY_REMOTE_ACTUATION` tetap `n`; branch ini tidak meminta command key.
4. `cldt_policy_guard_end_run()` mempertahankan identity sampai terminal record tersimpan, kemudian mengembalikan safe policy dan membersihkan run/authority/epoch.
5. `cldt_policy_guard_accept()` dan `cldt_policy_guard_fallback()` tetap tidak dipanggil pada baseline. TODO remote proposal, TTL, rejection, fallback, dan requalification menjadi pekerjaan Week 5; notebook tidak menandainya selesai.
6. Repo tidak mempunyai `tests/test_policy_guard.c`. Dengan larangan menambah file, tidak ada unit-test target yang boleh dikarang. Baseline pilot hanya membuktikan init, shadow begin-run, remote-disabled state, dan end-run melalui log/trace on-device; limitation test coverage dicatat apa adanya.
7. Bila resolved profile registry belum tersedia, guard init dan ready baseline tetap blocked. Nilai digest nol atau profile string buatan tidak dipakai sebagai bypass.

| Identifier repo | Contoh evidence baseline |
|---|---|
| `cldt_policy_guard_t.active_run_id` setelah begin | e.g. sama dengan reserved baseline run |
| `cldt_policy_guard_t.command_authority_boot_id` | e.g. sama dengan launcher process identity |
| `cldt_policy_guard_t.applied_epoch` | e.g. 0 |
| `cldt_policy_guard_t.remote_actuation_enabled` | e.g. false |
| `cldt_policy_guard_t.active_policy` | e.g. byte-identik dengan `safe_fallback` |
| `CONFIG_CLDT_GATEWAY_REMOTE_ACTUATION` | e.g. n |
| Proposal acceptance count | e.g. 0 |
| end-run identity clear setelah terminal trace | e.g. PASS |

## Step 8: Gateway Backhaul dan Egress Publisher
### Agregasi Observasi dan Publikasi MQTT/Backhaul (backhaul.h)

### Header dan Kconfig yang terhubung

#### `firmware/gateway/main/backhaul.h`

In [ ]:
%%writefile /content/cldt_scratch/backhaul.h
#ifndef CLDT_GATEWAY_BACKHAUL_H
#define CLDT_GATEWAY_BACKHAUL_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#include "cldt/cldt_types.h"
#include "gateway_provisioning.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_backhaul_command_fn)(
    void *context,
    const uint8_t *payload,
    size_t payload_bytes,
    bool retained,
    uint64_t received_local_us);

typedef struct {
    const cldt_gateway_credentials_t *credentials;
    cldt_backhaul_command_fn on_command;
    void *callback_context;
} cldt_backhaul_config_t;

esp_err_t cldt_backhaul_init(const cldt_backhaul_config_t *config);
esp_err_t cldt_backhaul_start(void);

/* Publishes immutable observation bytes; caller retains ownership. */
esp_err_t cldt_backhaul_publish_observation(
    cldt_run_id_t run_id,
    cldt_node_id_t node_id,
    const uint8_t *payload,
    size_t payload_bytes);

/* Starts a local-only HTTP endpoint for a pending manifest, never telemetry. */
esp_err_t cldt_backhaul_start_manifest_server(void);

esp_err_t cldt_backhaul_stop(void);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/gateway/main/gateway_runtime.h`

In [ ]:
%%writefile /content/cldt_scratch/gateway_runtime.h
#ifndef CLDT_GATEWAY_RUNTIME_H
#define CLDT_GATEWAY_RUNTIME_H

#include <stdbool.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/event_groups.h"
#include "freertos/queue.h"
#include "freertos/task.h"

#include "cldt/cldt_event_trace.h"
#include "cldt/cldt_types.h"
#include "policy_guard.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef enum {
    CLDT_GATEWAY_BOOT = 0,
    CLDT_GATEWAY_PROVISIONING,
    CLDT_GATEWAY_FORMING_THREAD,
    CLDT_GATEWAY_IDLE,
    CLDT_GATEWAY_WARMUP,
    CLDT_GATEWAY_MEASURING,
    CLDT_GATEWAY_COOLDOWN,
    CLDT_GATEWAY_FALLBACK,
    CLDT_GATEWAY_FAULT
} cldt_gateway_state_t;

typedef struct {
    cldt_node_id_t node_id;
    cldt_gateway_state_t state;
    /* Sole owner of active run, policy, epoch, and remote-actuation state. */
    cldt_policy_guard_t policy_guard;
    QueueHandle_t thread_rx_queue;
    QueueHandle_t observation_queue;
    QueueHandle_t command_queue;
    EventGroupHandle_t events;
    TaskHandle_t supervisor_task;
    TaskHandle_t aggregator_task;
    TaskHandle_t publisher_task;
    bool started;
} cldt_gateway_runtime_t;

/*
 * Initializes caller-owned state and all static RTOS objects. No task may run
 * and no radio may start before this function succeeds completely.
 */
esp_err_t cldt_gateway_runtime_init(cldt_gateway_runtime_t *runtime);

/* Starts tasks only after provisioning, RCP, Thread, and backhaul are ready. */
esp_err_t cldt_gateway_runtime_start(cldt_gateway_runtime_t *runtime);

/* Requests bounded shutdown at a run boundary; it must not delete live tasks. */
esp_err_t cldt_gateway_runtime_request_stop(cldt_gateway_runtime_t *runtime);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/gateway/main/Kconfig.projbuild`

In [ ]:
%%writefile /content/cldt_scratch/gateway_Kconfig.projbuild
menu "CLDT Gateway"

config CLDT_GATEWAY_NODE_ID
    int "Provisioning fallback node ID"
    range 1 4294967295
    default 1
    help
        Used only before a provisioned identity is present in NVS.

config CLDT_GATEWAY_RX_QUEUE_LENGTH
    int "Thread receive queue slots"
    range 4 128
    default 32
    help
        Bounded slots between Thread receive and aggregation. Select using peak
        observed arrival rate and consumer service time, then trace high-water.

config CLDT_GATEWAY_TRACE_QUEUE_LENGTH
    int "Observation queue slots"
    range 16 512
    default 128
    help
        Bounded observation queue. Overflow must become a visible health event;
        this setting is not permission to retain arbitrary broker backlog.

config CLDT_GATEWAY_MAX_TOTAL_RATE_PPS
    int "Compiled maximum aggregate application rate"
    range 1 500
    default 100
    help
        Compiled aggregate ceiling enforced by the edge guard. It is independent
        of host prediction and remains active during a host or broker failure.

config CLDT_GATEWAY_MAX_POLICY_TTL_MS
    int "Compiled maximum policy lifetime"
    range 1000 60000
    default 10000
    help
        Longest acceptable remote-policy lifetime. A control_profile may request
        a shorter TTL but cannot extend a command beyond this local bound.

config CLDT_GATEWAY_REMOTE_ACTUATION
    bool "Compile candidate remote-actuation path"
    default n
    help
        Keep disabled during testbed and shadow-model work. Enabling this switch
        does not bypass ready-manifest, authentication, run/epoch, TTL, local
        health, edge-limit, fallback, or endpoint validation requirements.

config CLDT_GATEWAY_RCP_UART_RX_GPIO
    int "RCP UART receive GPIO"
    range 0 48
    default 18
    help
        Gateway receive pin connected to the RCP transmit pin. Verify crossover,
        voltage compatibility, and the selected RCP UART configuration together.

config CLDT_GATEWAY_RCP_UART_TX_GPIO
    int "RCP UART transmit GPIO"
    range 0 48
    default 17
    help
        Gateway transmit pin connected to the RCP receive pin. Record final
        wiring, baud rate, and reset behavior in the upstream bring-up evidence.

config CLDT_GATEWAY_RCP_RESET_GPIO
    int "RCP reset GPIO"
    range -1 48
    default -1
    help
        -1 leaves optional RCP reset control disabled. Select a free S3 GPIO only
        after the C6 EN electrical behavior and wiring have been verified.

config CLDT_GATEWAY_RCP_BOOT_GPIO
    int "RCP boot GPIO"
    range -1 48
    default -1
    help
        -1 leaves optional RCP boot-mode control disabled. Treat any enabled
        value as board-specific wiring and validate it before automated reset.

endmenu


#### `firmware/gateway/CMakeLists.txt`

Root gateway file hanya membuka project ESP-IDF dan common component.

In [ ]:
%%writefile /content/cldt_scratch/gateway_CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(EXTRA_COMPONENT_DIRS "${CMAKE_CURRENT_LIST_DIR}/../../common")

include($ENV{IDF_PATH}/tools/cmake/project.cmake)
project(cldt_gateway)


#### `firmware/gateway/sdkconfig.defaults`

Defaults ini mengaktifkan trace, watchdog, BLE, OpenThread, MQTT, dan crypto support. Enablement pada build tidak membuktikan provisioning, bridge, broker, atau auth path bekerja.

In [ ]:
%%writefile /content/cldt_scratch/gateway_sdkconfig.defaults
CONFIG_FREERTOS_HZ=1000
CONFIG_FREERTOS_USE_TRACE_FACILITY=y
CONFIG_FREERTOS_GENERATE_RUN_TIME_STATS=y
CONFIG_ESP_TASK_WDT_EN=y
CONFIG_BT_ENABLED=y
CONFIG_BT_NIMBLE_ENABLED=y
CONFIG_OPENTHREAD_ENABLED=y
CONFIG_MQTT_PROTOCOL_311=y
CONFIG_MBEDTLS_CHACHAPOLY_C=y
CONFIG_MBEDTLS_CHACHA20_C=y
CONFIG_MBEDTLS_POLY1305_C=y


#### `firmware/gateway/sdkconfig.unicore`

File ini bukan tambahan baseline. Ia hanya diterapkan pada build directory terpisah untuk treatment SMP-versus-unicore yang masih deferred.

In [ ]:
%%writefile /content/cldt_scratch/gateway_sdkconfig.unicore
# Apply this file in a separate build directory for the SMP comparison.
CONFIG_FREERTOS_UNICORE=y


### Source TODO: `firmware/gateway/main/backhaul.c`

In [ ]:
%%writefile /content/cldt_scratch/backhaul.c
#include "backhaul.h"

esp_err_t cldt_backhaul_init(const cldt_backhaul_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate credentials and callbacks without logging
     * secrets, initialize Wi-Fi station mode, wait for private-LAN readiness,
     * create MQTT client state, and register callbacks that only enqueue bounded
     * command or connection events. Do not start a measured run, publish a policy,
     * or start the HTTP server from init. Every partial resource needs a defined
     * cleanup path for a failed credential, Wi-Fi, or broker connection.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_start(void)
{
    /*
     * IMPLEMENTATION TODO: start Wi-Fi and broker connections through an explicit
     * state machine, publish connection/health changes as traceable events, and
     * use bounded backoff outside any Thread, policy, or timing-critical lock.
     * A reconnect must never resurrect an expired policy or turn a retained broker
     * message into a command. The local guard remains safe while backhaul is down.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_publish_observation(
    cldt_run_id_t run_id,
    cldt_node_id_t node_id,
    const uint8_t *payload,
    size_t payload_bytes)
{
    (void)run_id;
    (void)node_id;
    (void)payload;
    (void)payload_bytes;

    /*
     * IMPLEMENTATION TODO: validate run/node IDs and bounded payload size, add
     * immutable envelope metadata, publish observations at the selected QoS, and
     * make disconnected behavior explicit: either bounded drop with a trace event
     * or a bounded local queue with a recorded high-water mark. Never block a
     * Thread receive task on broker reconnect and never use retained observations
     * as current truth for a controller.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_start_manifest_server(void)
{
    /*
     * IMPLEMENTATION TODO: bind only to the private experiment interface, cap
     * request/body size, accept a single complete manifest document into staging
     * storage, validate syntax and schema before acknowledgement, and pass only a
     * digest plus approved subset to supervisor. The HTTP handler may never alter
     * an active run. Reject credentials, command injection, and high-rate uploads.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_stop(void)
{
    /*
     * IMPLEMENTATION TODO: stop accepting HTTP uploads first, make command
     * callbacks reject new policy input, signal the supervisor that backhaul is
     * leaving service, drain or account for only the bounded observation queue,
     * disconnect MQTT and Wi-Fi with bounded timeouts, and clear callback state.
     * The function must be safe after a partial init and must not overwrite run
     * evidence just because the network is unavailable.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


### Source TODO: `firmware/gateway/main/gateway_runtime.c`

In [ ]:
%%writefile /content/cldt_scratch/gateway_runtime.c
#include "gateway_runtime.h"

esp_err_t cldt_gateway_runtime_init(cldt_gateway_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: reject a null runtime, clear state, load gateway node
     * identity, initialize the sole policy guard with its compiled safe policy,
     * and create all queues, event groups, trace storage, timer, and task stacks
     * statically. A new gateway boot has no resumable command-forwarding run;
     * remote actuation remains disarmed until a newly ledger-reserved run is
     * admitted.
     * Choose queue lengths from measured producer rates and document each owner.
     * Do not start Thread, Wi-Fi, BLE, broker, or any task here. An initialization
     * failure must leave the device in BOOT with no partially live project task.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_runtime_start(cldt_gateway_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: require successful init and all external prerequisites
     * (provisioning, RCP, Thread attach, backhaul readiness as required), start
     * supervisor first, verify admission of a newly ledger-reserved run, bind the
     * guard to it, then start aggregator and publisher under supervisor control.
     * Never resume authenticated command forwarding for a pre-reboot run. Give
     * each task a narrow ownership contract and measure stack margin before
     * choosing final priority/core affinity. If a later task fails, supervisor
     * stops earlier tasks in reverse order and emits a fault record rather than
     * continuing half-up.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_runtime_request_stop(cldt_gateway_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: set a supervisor event or task notification only. The
     * supervisor must stop manifest admission, command forwarding, and periodic
     * publication in a defined sequence, then request final endpoint counters and
     * emit final gateway status. Direct vTaskDelete from a caller would bypass
     * queue ownership and make accounting loss impossible to diagnose.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


### Entry point yang benar-benar ada: `firmware/gateway/main/app_main.c`

File ini belum memanggil runtime. Ia hanya mencetak peringatan scaffold dan menyimpan implementation order; jadi gateway project belum start, belum menerima frame, dan belum mempublikasikan observation.

In [ ]:
%%writefile /content/cldt_scratch/gateway_app_main.c
#include "esp_log.h"

static const char *TAG = "cldt_gateway";

void app_main(void)
{
    ESP_LOGW(TAG,
             "Research scaffold only: gateway runtime, provisioning, Thread "
             "bridge, backhaul, and policy guard are not implemented.");

    /*
     * IMPLEMENTATION ORDER:
     * 1. Prove provisioning storage and physical erase behavior without a run.
     * 2. Bring up the upstream RCP and Thread border router with no project policy.
     * 3. Add bounded Thread bridging and append-only observation publication.
     * 4. Add local policy guard, then only one expiring bulk-rate action.
     * 5. Compare SMP and unicore only after identical-build evidence exists.
     * Keep this entry point thin: construct runtime, call init/start, and let the
     * supervisor own failure handling. It must never become an ad-hoc demo flow.
     */
}


Pembagian ownership yang perlu muncul pada implementasi:

1. OpenThread callback hanya menaruh bounded copy ke `thread_rx_queue`.
2. Aggregator menjadi satu-satunya consumer `thread_rx_queue`: decode frame, cek run/node/CRC, gabungkan diagnostic snapshot yang tepat, lalu enqueue observation immutable ke `observation_queue`.
3. Publisher menjadi satu-satunya consumer `observation_queue` dan pemanggil `cldt_backhaul_publish_observation()`.
4. Disconnected behavior dipilih sekali: bounded drop yang terlihat atau bounded local queue. Tidak ada backlog tanpa batas.
5. QoS yang dipilih dicatat; source TODO memilih QoS 1 untuk durable observations. Observation tidak retained.
6. Current repo belum memiliki exact topic/envelope contract. Tentukan bytes dan topic di owner contract sebelum menulis publisher, lalu buat fixed encode/decode test. Jangan membuat key JSON ad hoc di beberapa module.
7. `cldt_gateway_runtime_request_stop()` hanya memberi notification. Supervisor menghentikan ingress, menguras atau meng-account queue, mengambil final status, lalu menutup backhaul.
8. `cldt_backhaul_start_manifest_server()` bukan jalur telemetry dan tidak dibuka selama measured run.
9. Command callback tetap reject/disabled karena `remote_actuation` false.

| Queue/state repo | Contoh pilot — ganti dengan hasil aktual |
|---|---|
| `cldt_gateway_runtime_t.thread_rx_queue` high-water | e.g. 7 dari 32 |
| `cldt_gateway_runtime_t.observation_queue` high-water | e.g. 11 dari 128 |
| Drop saat `thread_rx_queue` penuh | e.g. 0; bila nonzero tetap masuk evidence |
| Drop saat broker disconnected | e.g. 3; reason dan interval reconnect dicatat |
| `cldt_gateway_runtime_t.state` saat measurement | e.g. `CLDT_GATEWAY_MEASURING` |
| `cldt_gateway_runtime_t.started` setelah start | e.g. true |
| Observation retained | e.g. false |
| MQTT QoS | e.g. 1 |
| Waktu broker reconnect | e.g. 2.8 s |
| Stop setelah partial init | e.g. PASS |

## Common — Control Profile Admission
### Validasi Profil Kontrol dan Identitas Perlakuan (cldt_control_profile.h)

Ready baseline tetap menyebut `treatment.control_profile` walaupun `host_model` dan `remote_actuation` bernilai false. Validator portable dan test berikut memang sudah ada di repo; isinya dibiarkan sebagai TODO untuk dikerjakan pada source.

### `common/include/cldt/cldt_control_profile.h`

In [ ]:
%%writefile /content/cldt_scratch/cldt_control_profile.h
#ifndef CLDT_CONTROL_PROFILE_H
#define CLDT_CONTROL_PROFILE_H

#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * A ready manifest names one immutable control profile. The profile is resolved
 * by the host before a run starts, then its ID and digest are archived with the
 * evidence. It contains only safety-relevant selection values shared across
 * host and edge boundaries; it is not a generic configuration database.
 */
#define CLDT_CONTROL_PROFILE_ID_BYTES 65U
#define CLDT_CONTROL_PROFILE_DIGEST_BYTES 32U

typedef struct {
    /*
     * profile_id identifies the exact safety selection named by
     * treatment.control_profile in a ready manifest. calibration_id identifies
     * the separately versioned model-calibration evidence that supplies
     * residual and interval limits to the host fidelity gate.
     */
    char profile_id[CLDT_CONTROL_PROFILE_ID_BYTES];
    char calibration_id[CLDT_CONTROL_PROFILE_ID_BYTES];
    /* Version one permits actuation only from the frozen cross-layer candidate. */
    cldt_model_variant_t actuation_model_variant;

    /*
     * resolved_digest is the digest of the canonical, fully resolved profile
     * document. It prevents the same human-readable ID from silently referring
     * to different values in two evidence bundles. Digest calculation belongs
     * to the host registry/parser, not this portable validation function.
     */
    uint8_t resolved_digest[CLDT_CONTROL_PROFILE_DIGEST_BYTES];

    /* Host-side freshness and hysteresis inputs. */
    uint32_t maximum_observation_age_ms;
    uint16_t passing_windows_to_trust;

    /* Edge-side policy bounds. Compiled gateway/endpoints may be stricter. */
    uint32_t maximum_policy_ttl_ms;
    uint32_t maximum_total_rate_pps;
    uint32_t minimum_critical_period_ms;
    uint16_t maximum_bulk_burst_packets;
} cldt_control_profile_t;

/*
 * Validates only intrinsic profile shape and arithmetic safety. It performs no
 * file I/O, cryptographic digest calculation, model fitting, or device query.
 *
 * The later implementation must reject null/empty/non-terminated identifiers,
 * an all-zero digest, zero time/rate limits, and relationships that would make
 * a policy impossible to evaluate safely. It must leave caller-owned profile
 * bytes unchanged and return CLDT_ERR_NOT_IMPLEMENTED until those checks exist.
 */
cldt_status_t cldt_control_profile_validate(
    const cldt_control_profile_t *profile);

#ifdef __cplusplus
}
#endif

#endif


### `common/src/cldt_control_profile.c`

In [ ]:
%%writefile /content/cldt_scratch/cldt_control_profile.c
#include "cldt/cldt_control_profile.h"

cldt_status_t cldt_control_profile_validate(
    const cldt_control_profile_t *profile)
{
    (void)profile;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject a null profile. Verify profile_id and calibration_id contain a
     *    non-empty NUL-terminated identifier within their fixed-size buffers;
     *    never call an unbounded string function on data loaded from a file.
     * 2. Require a valid actuation_model_variant. Version-one actuated profiles
     *    must name CLDT_MODEL_CROSS_LAYER; a failed shadow acceptance test keeps
     *    actuation disabled rather than selecting a better-looking model later.
     * 3. Require resolved_digest to contain at least one nonzero byte. Digest
     *    verification itself belongs to the host registry because this common
     *    library deliberately has no JSON, filesystem, or cryptographic backend.
     * 4. Require nonzero observation age, trust-window count, TTL, rate ceiling,
     *    critical period, and bulk burst limit. Check any derived arithmetic
     *    with overflow-safe operations before returning success.
     * 5. Keep this function deterministic and side-effect free so a profile
     *    can be validated before recorder creation, network connection, or
     *    endpoint command issuance. Add unit tests for every rejection branch.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


### `tests/test_control_profile.c`

In [ ]:
%%writefile /content/cldt_scratch/test_control_profile.c
#include <stdio.h>

#include "cldt/cldt_control_profile.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Start with one completely specified in-memory profile whose IDs are
     *    bounded, digest is nonzero, and host/edge limits are all finite.
     * 2. Test one invalid condition at a time: null pointer, empty ID, missing
     *    NUL terminator, invalid/non-v1 actuation model, all-zero digest, zero
     *    freshness window, zero TTL, zero rate ceiling, zero critical period,
     *    and zero bulk burst ceiling.
     * 3. Assert exact status codes and assert the validator has not changed the
     *    input bytes. The test must not open a profile file or contact a device.
     * 4. Add a host-level test later for a manifest/profile-ID mismatch; that
     *    belongs above this portable common-library test.
     */
    fprintf(stderr, "SKIP: control profile tests have not been implemented.\n");
    return 77;
}


Pekerjaan yang dapat diselesaikan sepenuhnya pada tiga file tersebut:

1. `cldt_control_profile_validate()` memeriksa pointer, bounded NUL termination, non-empty `profile_id` dan `calibration_id`, nilai `actuation_model_variant`, digest yang tidak seluruhnya nol, serta seluruh finite nonzero limit.
2. Semua string diperiksa dengan bounded scan terhadap kapasitas array yang sudah dideklarasikan. Tidak ada `strlen()` pada bytes yang berasal dari file sebelum terminator terbukti berada di dalam buffer.
3. Derived arithmetic untuk rate/period/burst diperiksa overflow sebelum success. Validator tidak melakukan file I/O, digest calculation, network call, atau mutation.
4. Kalimat header tentang “version-one actuated profiles” dan signature validator yang tidak menerima flag actuation harus diberi satu interpretasi normatif di header/source/test. Notebook tidak mengubahnya menjadi rule baru secara diam-diam.
5. `test_control_profile.c` memakai satu struct valid lalu memutasi tepat satu kondisi per case. Setiap rejection branch memeriksa exact `cldt_status_t` dan input bytes tetap identik.
6. `return 77` diganti hanya setelah valid case, null, empty/nonterminated IDs, invalid model variant, zero digest, seluruh zero limit, dan arithmetic boundary diuji.

Ada satu dependency yang tidak dapat diselesaikan dengan tiga file itu saja: TODO di `host/main.c` meminta versioned local registry dan canonical profile document, tetapi repo pada commit acuan tidak mempunyai file registry atau format registry. Dengan batas “tidak menambah file/format”, notebook tidak boleh mengarang `profile_id`, `calibration_id`, atau digest. Konsekuensinya jelas: intrinsic validator dapat selesai dan pass, tetapi promotion `baseline.json` ke `state: "ready"` tetap tertahan sampai artefak registry yang nyata disetujui masuk repo.

| Identifier repo | Contoh evidence setelah implementasi |
|---|---|
| `cldt_control_profile_t.profile_id` | e.g. exact ID dari registry document yang benar-benar ada |
| `cldt_control_profile_t.calibration_id` | e.g. exact calibration identity dari document yang sama |
| `cldt_control_profile_t.resolved_digest` | e.g. 32 bytes hasil canonical document digest |
| `cldt_control_profile_t.actuation_model_variant` | e.g. value yang lolos rule versi satu yang sudah dibekukan |
| Input struct unchanged | e.g. PASS pada success dan seluruh rejection cases |
| Exit code `test_control_profile` | e.g. 0 |
| Profile registry resolution di `host/main.c` | e.g. BLOCKED sampai registry artifact benar-benar ada |
| Baseline manifest state sebelum resolution | e.g. template |

## Step 9: Host Manifest Admission
### Validasi Skema dan Pembekuan Parameter Eksperimen (experiment_config.h)

### Header/build contract

#### `host/experiment_config.h`

In [ ]:
%%writefile /content/cldt_scratch/experiment_config.h
#ifndef CLDT_HOST_EXPERIMENT_CONFIG_H
#define CLDT_HOST_EXPERIMENT_CONFIG_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * This is the executable subset of a manifest, not a mirror of every planning
 * field in JSON. A state == "template" document is intentionally incomplete and
 * must never be converted into this structure. Parse only a completed
 * state == "ready" manifest after JSON Schema validation.
 */
#define CLDT_MAX_MANIFEST_NODES 4U
#define CLDT_MAX_WORKLOADS 4U
#define CLDT_CONFIG_DIGEST_BYTES 32U
#define CLDT_MANIFEST_ID_BYTES 65U
#define CLDT_MANIFEST_TEXT_BYTES 161U

typedef enum {
    CLDT_TREATMENT_BASELINE = 0,
    CLDT_TREATMENT_PREDICTION,
    CLDT_TREATMENT_GATED_CONTROL,
    CLDT_TREATMENT_SAFETY,
    CLDT_TREATMENT_SMP,
    CLDT_TREATMENT_POWER
} cldt_treatment_mode_t;

typedef enum {
    CLDT_ACTION_NONE = 0,
    CLDT_ACTION_BULK_RATE_REDUCE,
    /* Reserved for explicitly admitted future phases; rejected by v1 control. */
    CLDT_ACTION_PHASE_STAGGER,
    CLDT_ACTION_POWER_PROFILE
} cldt_candidate_action_t;

typedef enum {
    CLDT_SCENARIO_NONE = 0,
    CLDT_SCENARIO_LOAD_STEP,
    CLDT_SCENARIO_OBSERVATION_PAUSE,
    CLDT_SCENARIO_ENDPOINT_RESTART,
    CLDT_SCENARIO_TOPOLOGY_SHIFT
} cldt_scenario_kind_t;

/*
 * Labels remain strings until provisioning maps them to physical numeric node
 * IDs. Do not hash labels ad hoc: an implementation must reject an unknown
 * label or use one documented collision-checked mapping.
 */
typedef struct {
    char label[CLDT_MANIFEST_ID_BYTES];
    cldt_node_role_t role;
} cldt_manifest_node_t;

typedef struct {
    char id[CLDT_MANIFEST_ID_BYTES];
    char source_label[CLDT_MANIFEST_ID_BYTES];
    cldt_traffic_class_t traffic_class;
    uint32_t period_ms;
    uint16_t payload_bytes;
    uint32_t deadline_ms;
    uint16_t burst_packets;
} cldt_workload_config_t;

typedef struct {
    /*
     * run_id is assigned by the launcher only after parsing and cross-field
     * validation. It is never taken from a template. Before assignment, the
     * launcher reserves a nonzero cryptographically generated value in the
     * durable global run ledger and binds a non-secret command-key identity only
     * for an actuated run. The parser leaves it zero; the coordinator rejects zero.
     */
    char experiment_id[CLDT_MANIFEST_ID_BYTES];
    cldt_run_id_t run_id;
    /* Nonzero identity of the launcher process; assigned beside run_id. */
    cldt_boot_id_t command_authority_boot_id;
    uint32_t seed;

    cldt_manifest_node_t nodes[CLDT_MAX_MANIFEST_NODES];
    size_t node_count;
    uint8_t thread_channel;
    char placement[CLDT_MANIFEST_TEXT_BYTES];
    char firmware_reference[CLDT_MANIFEST_TEXT_BYTES];

    uint32_t warmup_s;
    uint32_t measurement_s;
    uint32_t cooldown_s;
    uint16_t repetitions;

    cldt_workload_config_t workloads[CLDT_MAX_WORKLOADS];
    size_t workload_count;

    cldt_scenario_kind_t scenario;
    uint32_t scenario_at_s;
    uint32_t scenario_duration_s;
    char scenario_target[CLDT_MANIFEST_TEXT_BYTES];

    cldt_treatment_mode_t treatment_mode;
    cldt_candidate_action_t candidate_action;
    /*
     * Identifier selected by treatment.control_profile. Parsing preserves this
     * bounded name only; the host registry must resolve it to a
     * cldt_control_profile_t, verify its digest, and record both identities in
     * the evidence bundle before coordinator initialization.
     */
    char control_profile[CLDT_MANIFEST_TEXT_BYTES];
    bool host_model_enabled;
    bool remote_actuation_enabled;

    bool counter_reconciliation_required;
    double minimum_critical_on_time_pdr;
    char negative_case[CLDT_MANIFEST_TEXT_BYTES];
    uint8_t canonical_digest[CLDT_CONFIG_DIGEST_BYTES];
} cldt_experiment_config_t;

/*
 * Parses exactly one UTF-8 ready manifest from caller-owned bytes.
 *
 * Implementation sequence:
 * - validate JSON syntax and schema first;
 * - reject state == "template" before allocating or opening I/O;
 * - copy bounded fields, preserving a precise error path;
 * - calculate canonical_digest after full validation and leave run_id plus
 *   command_authority_boot_id zero for the launcher's separate assignment step.
 *
 * The parser must reject unknown runtime fields, duplicate object keys, secrets,
 * and any null value that a ready manifest is required to replace.
 */
cldt_status_t cldt_experiment_config_parse(
    const char *json,
    size_t json_bytes,
    cldt_experiment_config_t *output);

/*
 * Performs deterministic cross-field validation after parsing.
 *
 * It verifies node/stream references, rate and deadline feasibility, scenario
 * timing, treatment permissions, and compiled safety ceilings. It must not make
 * network calls, create a directory, or mutate output state.
 */
cldt_status_t cldt_experiment_config_validate(
    const cldt_experiment_config_t *config);

#ifdef __cplusplus
}
#endif

#endif


#### `host/CMakeLists.txt`

In [ ]:
%%writefile /content/cldt_scratch/host_CMakeLists.txt
add_executable(cldt_host
    main.c
    coordinator.c
    experiment_config.c
    twin_model.c
    estimator.c
    fidelity_gate.c
    policy.c
    broker_io.c
    run_recorder.c
    kalman.c
)

target_include_directories(cldt_host PRIVATE ${CMAKE_CURRENT_SOURCE_DIR})
target_link_libraries(cldt_host PRIVATE cldt_common)

if(MSVC)
    target_compile_options(cldt_host PRIVATE /W4)
else()
    target_compile_options(cldt_host PRIVATE -Wall -Wextra -Wpedantic -Wconversion)
endif()


### Source TODO: `host/experiment_config.c`

In [ ]:
%%writefile /content/cldt_scratch/experiment_config.c
#include "experiment_config.h"

cldt_status_t cldt_experiment_config_parse(
    const char *json,
    size_t json_bytes,
    cldt_experiment_config_t *output)
{
    (void)json;
    (void)json_bytes;
    (void)output;

    /*
     * IMPLEMENTATION TODO:
     * 1. Use a maintained JSON parser with a bounded input limit; parse exactly
     *    one UTF-8 document and reject duplicate keys rather than accepting a
     *    library-specific last-key-wins behavior.
     * 2. Validate against schemas/experiment.schema.json before conversion.
     *    This runtime parser accepts only state == "ready"; a template is a
     *    planning artifact and must never start a physical run.
     * 3. Copy strings into fixed, NUL-terminated fields only after checking the
     *    destination capacity. Require a non-empty control_profile for a ready
     *    run and preserve JSON Pointer-like error paths for the operator instead
     *    of returning a generic parse failure.
     * 4. Compute the canonical manifest digest from the original validated bytes
     *    using one documented canonicalization rule; do not include credentials.
     *    Leave output.run_id and output.command_authority_boot_id zero. The
     *    launcher assigns them only after a separate durable global-ledger
     *    reservation; parsing must not silently create nonce state or require a
     *    command key for a shadow-only run.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_experiment_config_validate(
    const cldt_experiment_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate cross-field relationships that schema shape
     * checks cannot prove: every stream source must name an endpoint, deadlines
     * must be compatible with their period, aggregate offered load must fit the
     * compiled safety ceiling, scenario time must lie inside measurement time,
     * and remote actuation must be disabled for non-control treatments. Version
     * one accepts only NONE or BULK_RATE_REDUCE and rejects the reserved phase
     * and power actions even though planning templates can name them. Reject a
     * configuration before any adapter or run directory is opened. Keep this
     * function deterministic so the same manifest has the same outcome on host
     * and in future gateway subset validation.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


### Entry point TODO: `host/main.c`

In [ ]:
%%writefile /content/cldt_scratch/test_auth.c
#include <stdio.h>
#include <stdlib.h>

#include "coordinator.h"

int main(int argc, char **argv)
{
    (void)argc;
    (void)argv;

    fprintf(stderr,
            "CLDT host scaffold: implement manifest loading, recording, "
            "modeling, and fidelity-gated coordination before use.\n");

    /*
     * IMPLEMENTATION TODO:
     * 1. Accept exactly one manifest path and one optional results-root path;
     *    print a short usage error for every other argument shape.
     * 2. Read with a bounded size, validate the JSON schema, and refuse a
     *    state == "template" manifest before any network connection is made.
     * 3. Resolve the manifest's named control profile from a versioned local
     *    registry, verify its canonical digest, and reject absent or mismatched
     *    profile/calibration identities before opening a broker connection.
     * 4. Reserve a nonzero CSPRNG run ID in the durable global run ledger,
     *    collision-check it, assign it to the validated config, and generate a
     *    nonzero coordinator-process boot ID. If remote actuation is requested,
     *    also bind the non-secret command-key identity;
     *    if ledger continuity is unavailable, require key rotation before
     *    actuation. Never guess or reuse an ID. Shadow-only runs need no key.
     * 5. Create the immutable run directory, install signal handling that only
     *    requests a stop, initialize the coordinator with both config and
     *    resolved profile, and enter its event loop.
     * 6. Return a nonzero status for invalid, interrupted, or failed runs. A
     *    successful process exit is not evidence that a result is valid.
     */
    return EXIT_FAILURE;
}


Pekerjaan host:

1. Satu maintained JSON parser/schema validator dipilih dengan bounded input dan duplicate-key rejection. Hanya target library yang benar-benar dipilih yang ditambahkan ke `host/CMakeLists.txt`; notebook tidak menulis nama dependency placeholder.
2. Canonicalization rule dan SHA-256 backend dipilih sekali untuk `canonical_digest`, manifest archive, dan profile digest. Exact input bytes serta perbedaan antara hash raw bytes dan canonical document ditulis normatif sebelum implementation.
3. `cldt_experiment_config_parse()` menerima hanya schema branch `ready`, menyalin string secara bounded, menghitung digest menurut rule tersebut, dan meninggalkan `run_id` serta `command_authority_boot_id` bernilai 0.
4. `cldt_experiment_config_validate()` memeriksa source endpoint, period/deadline, total offered load, scenario timing, dan treatment permissions tanpa I/O atau mutation.
5. Launcher mereservasi nonzero `run_id` melalui CSPRNG dan durable ledger sebelum recorder/network. Karena tree belum mempunyai owner/format ledger, kontrak itu tetap unresolved sampai ditempatkan secara nyata; nilai tidak ditebak atau disimpan hanya dalam RAM.
6. Ready manifest memerlukan `treatment.control_profile`; profile registry document/digest harus benar-benar tersedia sebelum coordinator init. String ID tanpa registry tidak membuat manifest eligible.
7. `main()` menerima satu manifest path dan satu optional results root, memasang signal handler yang hanya meminta stop, lalu meneruskan config/profile yang sudah admitted. Argument lain menghasilkan usage error.
8. `treatment.host_model == false` dan `treatment.remote_actuation == false` menentukan baseline branch. Branch ini tidak meminta command key dan tidak menginisialisasi model/gate sebagai prasyarat recorder.

| Field repo | Contoh format |
|---|---|
| `cldt_experiment_config_t.experiment_id` | e.g. stable-baseline |
| `cldt_experiment_config_t.run_id` setelah ledger reservation | e.g. nonzero 64-bit value dari CSPRNG dan collision check |
| `cldt_experiment_config_t.command_authority_boot_id` | e.g. nonzero 32-bit process identity |
| `cldt_experiment_config_t.thread_channel` | e.g. 15 |
| `cldt_experiment_config_t.host_model_enabled` | e.g. false |
| `cldt_experiment_config_t.remote_actuation_enabled` | e.g. false |
| `cldt_experiment_config_t.canonical_digest` | e.g. 32 bytes yang cocok dengan archived manifest digest |
| Template input | e.g. `CLDT_ERR_NOT_READY` |
| Duplicate JSON key | e.g. rejected sebelum conversion |
| Unknown stream source | e.g. rejected oleh `cldt_experiment_config_validate()` |

## Step 10: Host Append-Only Recorder
### Perekaman Raw Bytes dan Output events.ndjson (run_recorder.h)

### Header contract: `host/run_recorder.h`

In [ ]:
%%writefile /content/cldt_scratch/run_recorder.h
#ifndef CLDT_HOST_RUN_RECORDER_H
#define CLDT_HOST_RUN_RECORDER_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>
#include <stdio.h>

#include "cldt/cldt_status.h"
#include "experiment_config.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    /* event_stream is append-only. finalized prevents a second terminal status. */
    FILE *event_stream;
    char run_directory[260];
    uint64_t records_written;
    bool finalized;
} cldt_run_recorder_t;

/*
 * Creates a new run directory and fails if it already exists. The recorder owns
 * only files it creates for this run; it may never delete or rewrite a prior run.
 */
cldt_status_t cldt_run_recorder_open(
    cldt_run_recorder_t *recorder,
    const char *results_root,
    const cldt_experiment_config_t *config,
    const char *original_manifest,
    size_t original_manifest_bytes);

/* Appends one raw received record; parsing and model updates happen elsewhere. */
cldt_status_t cldt_run_recorder_append(
    cldt_run_recorder_t *recorder,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint64_t received_host_us);

/* Writes one immutable terminal status after counters and evidence are collected. */
cldt_status_t cldt_run_recorder_finalize(
    cldt_run_recorder_t *recorder,
    const char *status,
    const char *reason);

#ifdef __cplusplus
}
#endif

#endif


### Source TODO: `host/run_recorder.c`

In [ ]:
%%writefile /content/cldt_scratch/run_recorder.c
#include "run_recorder.h"

cldt_status_t cldt_run_recorder_open(
    cldt_run_recorder_t *recorder,
    const char *results_root,
    const cldt_experiment_config_t *config,
    const char *original_manifest,
    size_t original_manifest_bytes)
{
    (void)recorder;
    (void)results_root;
    (void)config;
    (void)original_manifest;
    (void)original_manifest_bytes;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject a template config, zero/unreserved run ID, unsafe path component,
     *    empty manifest, or pre-existing target directory. Derive the directory
     *    name from the reserved run identifier plus timestamp, never from
     *    unchecked user input. Record the coordinator boot ID, ledger identity,
     *    and, for actuation, the non-secret command-key identity so nonce
     *    provenance is auditable.
     * 2. Create the directory atomically, copy the original manifest byte-for-
     *    byte, write its SHA-256 and a versions placeholder that will resolve
     *    source/binary identities plus the selected control_profile, then fsync
     *    metadata before accepting observations.
     * 3. Open events.ndjson in append-only mode and leave finalized false. If any
     *    step fails, roll back only the newly created empty directory.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_run_recorder_append(
    cldt_run_recorder_t *recorder,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint64_t received_host_us)
{
    (void)recorder;
    (void)topic;
    (void)payload;
    (void)payload_bytes;
    (void)received_host_us;

    /*
     * IMPLEMENTATION TODO: require an open non-finalized recorder, validate a
     * bounded topic and payload length, encode binary payload safely (for example
     * base64), escape all JSON strings, and append exactly one newline-terminated
     * record containing host receive time. Flush according to a documented
     * durability policy and increment records_written only after a successful
     * write. Do not parse, reorder, or discard raw evidence in this layer.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_run_recorder_finalize(
    cldt_run_recorder_t *recorder,
    const char *status,
    const char *reason)
{
    (void)recorder;
    (void)status;
    (void)reason;

    /*
     * IMPLEMENTATION TODO: accept only a fixed status vocabulary and predefined
     * exclusion reasons, write one small run-status.json atomically, flush and
     * close the event stream, then set finalized true. A second finalize call
     * must fail without changing files. Never reopen or rewrite events.ndjson
     * during finalization, even when the run is invalid or interrupted.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Pekerjaan fungsi:

1. `cldt_run_recorder_open()` menolak template, zero/unreserved run ID, path component tidak aman, empty manifest, dan target directory yang sudah ada.
2. Directory name hanya berasal dari reserved run identity dan timestamp yang disanitasi. Existing directory tidak pernah dipakai ulang.
3. Original manifest ditulis byte-for-byte, digest disimpan, lalu `versions.json` mulai diisi sebelum observation diterima.
4. `events.NDJSON` dibuka append-only. Exact NDJSON record contract harus dibekukan karena TODO baru menyebut topic, binary payload, dan host receive time; notebook tidak menambah key yang tidak ada.
5. Binary payload memakai encoding aman dan reversible. Recorder tidak decode, reorder, deduplicate, atau membuang.
6. `records_written` bertambah hanya setelah newline record benar-benar berhasil ditulis sesuai durability policy.
7. `cldt_run_recorder_finalize()` menerima fixed status vocabulary, menulis satu `run-status.json` secara atomik, menutup stream, lalu menetapkan `finalized`.
8. Test non-reportable wajib memutus proses setelah beberapa record. Hasil yang benar adalah terminal status interrupted dan raw rows tetap ada.
9. C11 tidak menyediakan atomic directory/fsync portability lengkap. `run_recorder.c` harus menyatakan platform host yang didukung atau mengisolasi operation OS secara jelas; jangan menganggap `fflush()` sama dengan durable metadata.

| Artifact/field repo | Contoh format — ganti dengan hasil aktual |
|---|---|
| `cldt_run_recorder_t.run_directory` | e.g. results\20260930-101500-  |
| Frozen manifest | e.g. manifest.json, bytes identik dengan input ready file |
| Manifest digest | e.g. manifest.sha256, 64 karakter hexadecimal |
| Versions record | e.g. versions.json |
| `cldt_run_recorder_t.event_stream` | e.g. open handle untuk events.NDJSON |
| `cldt_run_recorder_t.records_written` | e.g. 1200 |
| `cldt_run_recorder_t.finalized` sebelum finalize | e.g. false |
| Terminal file | e.g. run-status.json |
| Status normal | e.g. complete |
| Status interruption pilot | e.g. interrupted |
| Second finalize | e.g. rejected; file lama tidak berubah |

## Step 11: Host Broker IO dan Baseline Coordinator
### Integrasi Broker Client dan Loop Baseline Tanpa Model Aktif

### Header contracts

#### `host/broker_io.h`

In [ ]:
%%writefile /content/cldt_scratch/broker_io.h
#ifndef CLDT_HOST_BROKER_IO_H
#define CLDT_HOST_BROKER_IO_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_broker_message_fn)(
    void *context,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint64_t received_host_us,
    bool retained);

typedef struct {
    void *native_client;
    cldt_broker_message_fn on_message;
    void *callback_context;
    bool connected;
} cldt_broker_io_t;

cldt_status_t cldt_broker_io_open(
    cldt_broker_io_t *io,
    const char *host,
    uint16_t port,
    cldt_broker_message_fn callback,
    void *callback_context);

cldt_status_t cldt_broker_io_poll(cldt_broker_io_t *io, uint32_t timeout_ms);

cldt_status_t cldt_broker_io_publish(
    cldt_broker_io_t *io,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint8_t qos,
    bool retained);

void cldt_broker_io_close(cldt_broker_io_t *io);

#ifdef __cplusplus
}
#endif

#endif


#### `host/coordinator.h`

In [ ]:
%%writefile /content/cldt_scratch/coordinator.h
#ifndef CLDT_HOST_COORDINATOR_H
#define CLDT_HOST_COORDINATOR_H

#include <stdbool.h>
#include <stdint.h>

#include "broker_io.h"
#include "cldt/cldt_control_profile.h"
#include "estimator.h"
#include "experiment_config.h"
#include "fidelity_gate.h"
#include "policy.h"
#include "run_recorder.h"
#include "twin_model.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    cldt_experiment_config_t config;
    /*
     * Resolved profile copied only after its ID matches config.control_profile
     * and its immutable digest has been recorded by the run recorder.
     */
    cldt_control_profile_t control_profile;
    cldt_broker_io_t broker;
    cldt_run_recorder_t recorder;
    /* One frozen instance and estimator per declared comparison variant. */
    cldt_twin_model_t models[CLDT_MODEL_VARIANT_COUNT];
    cldt_estimator_t estimators[CLDT_MODEL_VARIANT_COUNT];
    cldt_fidelity_gate_t gate;
    cldt_policy_t active_policy;
    uint64_t started_host_us;
    bool stop_requested;
} cldt_coordinator_t;

cldt_status_t cldt_coordinator_init(
    cldt_coordinator_t *coordinator,
    const cldt_experiment_config_t *config,
    const cldt_control_profile_t *control_profile);

cldt_status_t cldt_coordinator_run(cldt_coordinator_t *coordinator);

void cldt_coordinator_request_stop(cldt_coordinator_t *coordinator);

#ifdef __cplusplus
}
#endif

#endif


### Source TODO: `host/broker_io.c`

In [ ]:
%%writefile /content/cldt_scratch/broker_io.c
#include "broker_io.h"

cldt_status_t cldt_broker_io_open(
    cldt_broker_io_t *io,
    const char *host,
    uint16_t port,
    cldt_broker_message_fn callback,
    void *callback_context)
{
    (void)io;
    (void)host;
    (void)port;
    (void)callback;
    (void)callback_context;

    /*
     * IMPLEMENTATION TODO: validate host, port, callback, and context; create a
     * libmosquitto (or equally maintained) client with explicit protocol version,
     * TLS configuration when used, and a stable reconnect state machine. Subscribe
     * only to the private experiment namespace. The callback must enqueue or
     * return quickly; it must not update the model or write files directly.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_broker_io_poll(cldt_broker_io_t *io, uint32_t timeout_ms)
{
    (void)io;
    (void)timeout_ms;

    /*
     * IMPLEMENTATION TODO: validate io and cap timeout_ms to a small documented
     * value so coordinator deadlines are observed even during a quiet broker.
     * Translate broker reconnect, protocol, and backpressure conditions into
     * explicit CLDT statuses. Retained command delivery is a safety event, not a
     * convenient reconnect feature. Poll must never spin indefinitely.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_broker_io_publish(
    cldt_broker_io_t *io,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint8_t qos,
    bool retained)
{
    (void)io;
    (void)topic;
    (void)payload;
    (void)payload_bytes;
    (void)qos;
    (void)retained;

    /*
     * IMPLEMENTATION TODO: reject null/oversized input and out-of-range QoS;
     * require QoS 1 for durable observations as selected by the final design;
     * reject retained payloads on command topics; and expose publish queue or
     * reconnect pressure to the coordinator. The broker adapter must not invent
     * retries that extend a policy past its TTL.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

void cldt_broker_io_close(cldt_broker_io_t *io)
{
    (void)io;

    /*
     * IMPLEMENTATION TODO: unsubscribe, disconnect with a bounded deadline,
     * destroy only the native client owned by io, clear callbacks and connected
     * state, and leave raw run files to the recorder. Closing the broker must be
     * safe after a partial open and must not make a disconnected run look valid.
     */
}


### Source TODO: `host/coordinator.c`

In [ ]:
%%writefile /content/cldt_scratch/coordinator.c
#include "coordinator.h"

cldt_status_t cldt_coordinator_init(
    cldt_coordinator_t *coordinator,
    const cldt_experiment_config_t *config,
    const cldt_control_profile_t *control_profile)
{
    (void)coordinator;
    (void)config;
    (void)control_profile;

    /*
     * IMPLEMENTATION TODO:
     * 1. Require a ready, cross-field-validated config with a nonzero run ID
     *    already reserved in the durable global run ledger and a nonzero command
     *    authority boot ID, plus a valid resolved control profile. Compare the
     *    profile ID with config.control_profile
     *    using bounded strings; reject mismatch before opening a recorder or
     *    broker. The caller is responsible for checking the profile digest
     *    against the canonical registry document before this function is called.
     * 2. Copy config and profile only after validation. Record the profile ID,
     *    calibration ID, actuation-model variant, and digest beside the frozen
     *    manifest. Version one may name only the cross-layer variant for
     *    actuation; failed shadow acceptance means no actuation, not model swap.
     *    Initialize the fidelity gate and edge proposal limits from that
     *    immutable selection.
     * 3. Initialize recorder, all three model/estimator pairs, fidelity gate,
     *    policy baseline, and broker in that order. Each successful step needs a
     *    paired rollback action so a later failure leaves no partial run marked
     *    valid.
     * 4. Do not open a network adapter before the immutable run directory,
     *    manifest digest, and control-profile identity exist.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_coordinator_run(cldt_coordinator_t *coordinator)
{
    (void)coordinator;

    /*
     * IMPLEMENTATION TODO: implement one bounded event loop with this strict
     * sequence for each accepted observation: record raw bytes first; validate
     * run/digest identity; update each model only from its allowed features;
     * score all variants on identical completed prior horizons; evaluate the
     * fidelity gate only for the frozen actuation variant; and only then consider
     * a new bounded proposal. Sleep or poll with a deadline so policy expiry,
     * phase transitions, and stop requests are never starved by broker traffic.
     * Finalize through the recorder with complete, invalid, or interrupted status.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

void cldt_coordinator_request_stop(cldt_coordinator_t *coordinator)
{
    (void)coordinator;

    /*
     * IMPLEMENTATION TODO: make this function only set an atomic or signal-safe
     * stop flag. The main loop owns broker close, final counter requests, metric
     * reconciliation, and recorder finalization because those operations may
     * allocate, block, or fail. A signal handler must never write evidence or
     * publish a fallback command directly.
     */
}


Baseline path Week 3:

1. Satu maintained MQTT client untuk host dipilih, lalu actual include/link target-nya ditambahkan ke `host/CMakeLists.txt`. Repo belum memilih client tersebut; nama library placeholder tidak ditulis pada notebook.
2. `cldt_broker_io_open()` hanya subscribe private experiment namespace. Callback menyalin/enqueue atau kembali cepat.
3. `cldt_broker_io_poll()` mempunyai bounded `timeout_ms`; quiet broker tidak boleh membuat stop, cooldown, atau finalization terlambat.
4. Retained observation ditandai invalid/stale input—bukan current physical truth.
5. Pada callback/event loop, `cldt_run_recorder_append()` dipanggil sebelum decode atau keputusan lain.
6. Raw frame kemudian di-decode dengan `cldt_protocol_decode()`, run/node identity diperiksa, dan trace record diteruskan ke audit/accounting path.
7. Karena `host_model_enabled` false, baseline path tidak menginisialisasi/update model sebagai prasyarat observation recording. Karena `remote_actuation_enabled` false, `cldt_broker_io_publish()` tidak dipakai untuk command.
8. Komentar TODO pada `cldt_coordinator_init()` menyebut seluruh model/gate dalam urutan init. Baseline branch memakai dua boolean config yang sudah ada sehingga pekerjaan Week 4/5 tidak menjadi syarat untuk merekam observation Week 3.
9. Stop signal hanya mengubah `stop_requested`; event loop menutup broker, mengambil final counters, menjalankan reconciliation, lalu finalize.
10. Publish command, retained command, dan broker retry yang memperpanjang TTL tetap tidak aktif.

| Field/status repo | Contoh hasil |
|---|---|
| `cldt_broker_io_t.connected` setelah open | e.g. true |
| `timeout_ms` pada poll | e.g. 100 |
| Raw append sebelum decode | e.g. PASS — urutan log menunjukkan recorder lebih dulu |
| Retained observation | e.g. invalid input; raw bytes tetap tersimpan |
| Wrong `run_id` | e.g. exact failure status dibekukan dalam test; raw bytes tetap tersimpan |
| CRC failure | e.g. exact failure status dibekukan dalam test; raw bytes tetap tersimpan |
| `cldt_coordinator_t.stop_requested` setelah signal | e.g. true |
| Model update count | e.g. 0 |
| Command publish count | e.g. 0 |
| Broker disconnect interval | e.g. tercatat dan run diklasifikasikan sesuai frozen rule |

## Step 12: Lifecycle Replay dan Rekonsiliasi Audit
### Eksekusi analysis/reproduce.py dan Verifikasi Konsistensi Bukti

`host/analysis/reproduce.py` tidak mempunyai pasangan `.h`; contract C yang dipanggil untuk dua lapis pemeriksaan tetap `common/include/cldt/cldt_metrics.h`, yang sudah ditempel utuh pada notebook Week 2.

In [ ]:
%%writefile /content/cldt_scratch/reproduce.py
import sys
import json
from pathlib import Path

def main():
    if len(sys.argv) != 2:
        print("Usage: python reproduce.py <results_dir>")
        sys.exit(1)
        
    # TODO: load manifest JSON from results_dir / "manifest.json"
    # TODO: verify manifest has state="ready" and all _todo items resolved
    # TODO: compute SHA-256 digest of manifest and compare against results_dir / "manifest.sha256"
    # TODO: load events.ndjson one JSON object per line; reject malformed, blank,
    # duplicate, or trailing non-JSON records
    # TODO: group events by (run_id, node_id, boot_id, sequence) for per-item lifecycle audit
    # TODO: for each lifecycle group, verify exactly one release event and one terminal event (ack/expire/drop)
    # TODO: count duplicate_releases, duplicate_terminals, terminal_without_release, unresolved_items
    # TODO: fit naive moving-average baseline on calibration data only
    # TODO: fit network-only model: features = [delivery_outcome, link_rssi, traffic_load]
    # TODO: fit cross-layer model from the frozen network, MAC, queue, and RTOS
    # feature allowlist
    # TODO: read manifest-defined calibration and held-out blocks; never invent a percentage split after seeing results
    # TODO: score all three models on identical held-out horizons: relative P95 error on deadline delivery ratio
    # TODO: compute primary uncertainty from run-level summaries or a whole-run cluster bootstrap
    # TODO: use within-run block bootstrap only for paired time-series uncertainty,
    # never as independent physical replication
    # TODO: compute prediction interval coverage: fraction of observations within predicted +/- 2 sigma
    # TODO: build a calibration-only support envelope and retain inside/outside status for every scored horizon
    # TODO: retain observation-integrity status; missing/stale/unreconciled horizons must not disappear silently
    # TODO: perform feature-group ablation only after the primary three-model comparison is frozen
    # TODO: generate gate characterization: state/reason vs time, trust fraction,
    # false trust, abstention/requalification latency, and P[2][2]
    # TODO: output the frozen primary metric table as CSV
    # TODO: exit nonzero if reconciliation fails (any lifecycle inconsistency)
    # TODO: use numpy for statistics, matplotlib for plots, scipy.stats for bootstrap

    print("ERROR: reproduction pipeline is a scaffold and produced no result.", file=sys.stderr)
    raise SystemExit(2)

if __name__ == "__main__":
    main()


Pada Week 3, hanya TODO yang menyusun evidence integrity yang dibuka:

1. Load manifest dari `results_dir / "manifest.json"`.
2. Verifikasi `state == "ready"` dan digest terhadap `manifest.sha256`.
3. Baca `events.NDJSON` line-by-line; blank, malformed, truncated, dan duplicate raw line tidak dihapus.
4. Decode payload sesuai exact recorder/envelope contract yang sudah dibekukan.
5. Kelompokkan work-item berdasarkan `run_id`, `node_id`, `boot_id`, dan `sequence`.
6. Pastikan tepat satu `CLDT_EVENT_TASK_RELEASE`, maksimal satu terminal, tidak ada terminal tanpa release, dan unresolved tidak tersembunyi.
7. Bandingkan hasil item audit dengan `cldt_metrics_reconcile()` per traffic class.
8. Exit nonzero bila lifecycle atau aggregate reconciliation gagal.

TODO moving average, network-only, cross-layer, calibration/held-out scoring, bootstrap, prediction interval, support envelope, ablation, dan gate characterization tetap belum diisi sampai Week 4/5. Jika satu fungsi belum dapat memberi lifecycle result tanpa membuat model palsu, CLI tetap nonzero; Week 3 closure harus memakai hasil audit yang benar-benar dapat diulang, bukan mengubah exit code saja.

| Pemeriksaan replay | Contoh hasil |
|---|---|
| Manifest bytes/digest | e.g. PASS |
| NDJSON malformed line | e.g. 0; bila nonzero run invalid |
| `cldt_item_audit_t.duplicate_releases` | e.g. 0 |
| `cldt_item_audit_t.duplicate_terminals` | e.g. 0 |
| `cldt_item_audit_t.terminal_without_release` | e.g. 0 |
| `cldt_item_audit_t.unresolved_items` | e.g. 0 pada terminal snapshot |
| `cldt_item_audit_t.consistent` | e.g. true |
| Semua `cldt_reconciliation_t.consistent` | e.g. true |
| Lifecycle replay exit code | e.g. 0 |
| Model table pada Week 3 | e.g. not produced |

## Step 13: Template Manifest Stable Baseline
### Konfigurasi Runtime experiments/baseline.json

### Strict runtime template: `experiments/baseline.json`

In [ ]:
%%writefile /content/cldt_scratch/baseline.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stable-baseline",
  "title": "Stable Thread Baseline",
  "purpose": {
    "question": "What does normal deadline delivery look like on the physical Thread topology before modelling or policy changes?",
    "comparison": "Repeated static runs with the same workload, no host model, and no remote actuation.",
    "primary_metric": "Reconciled on-time delivery ratio for the critical stream."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Freeze the real four-board topology only after upstream RCP and border-router attachment are repeatable.",
      "method": "Label the S3 gateway, RCP C6, router-capable C6, and low-power C6; survey the ordinary work area; select one 802.15.4 channel; mark board position, orientation, power path, and cables; then record actual Thread roles and binary identities.",
      "done_when": "The same configuration survives power cycling, IPv6 reachability is confirmed, and placement plus role evidence are archived before a reportable run."
    },
    {
      "path": "/execution",
      "action": "Find a stable workload and run length that characterize normal variance without hiding queue behavior.",
      "method": "Use three pilot repetitions with one critical and one telemetry stream. Adjust only one offered-load dimension at a time until the queue remains bounded, all counters reconcile, and the measurement window contains enough critical releases to make a service ratio meaningful.",
      "done_when": "The ready file has frozen stream details, phase durations, seed, and repetition count derived from a traceable pilot decision."
    },
    {
      "path": "/acceptance",
      "action": "Set an engineering service floor and raw-evidence requirement before calibration data is used by any model.",
      "method": "Choose a numerical critical on-time floor stricter than pilot noise, require reconciliation, set static/no-model/no-actuation treatment values, and require manifest, versions, topology, events, final counters, and operator notes in evidence.",
      "done_when": "Every baseline run can be classified complete, invalid, or interrupted without looking at a later model result."
    }
  ]
}


### Planning companion: `experiments/authoring/baseline.JSONC`

In [ ]:
%%writefile /content/cldt_scratch/baseline.jsonc
{
  // Authoring copy only. Keep the matching ../baseline.json as strict JSON.
  // Do not set state to ready until every null below has a measured or frozen value.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stable-baseline",
  "title": "Stable Thread Baseline",
  "purpose": {
    // Preserve the question. If the question changes, create a new experiment ID.
    "question": "What does normal deadline delivery look like on the physical Thread topology before modelling or policy changes?",
    "comparison": "Repeated static runs with the same workload, no host model, and no remote actuation.",
    "primary_metric": "Reconciled on-time delivery ratio for the critical stream."
  },
  "setup": {
    // Replace with exactly the four physical roles and their labels after repeated attachment works.
    "nodes": null,
    // Choose one surveyed IEEE 802.15.4 channel; do not select it from a favorable result.
    "thread_channel": null,
    // Describe positions, orientation, power path, cables, and the named baseline evidence block.
    "placement": null,
    // Record source revision, ESP-IDF/upstream revision, sdkconfig hash, and binary hashes.
    "firmware_reference": null
  },
  "execution": {
    // Warm-up is excluded from the primary calculation but must be recorded.
    "warmup_s": null,
    // Choose a long enough measured window to capture normal variance and many critical releases.
    "measurement_s": null,
    // Cooldown ends only after final counter and trace collection is requested.
    "cooldown_s": null,
    // Freeze the planned count before examining final baseline results.
    "repetitions": null,
    // Record the deterministic workload seed or an explicitly documented seed plan.
    "seed": null
  },
  "traffic": {
    // Add concrete critical and telemetry streams after pilots. Baseline has no host policy.
    "streams": null
  },
  "scenario": {
    // Stable baseline uses event none and a non-empty target such as "no injected disturbance".
    "event": null,
    "at_s": null,
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Use baseline mode, candidate_action none, host_model false, remote_actuation false.
    "mode": null,
    "candidate_action": null,
    // Name the immutable baseline profile even when it cannot actuate; archive its resolved digest.
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    // Must be true for every reportable result.
    "counter_reconciliation": null,
    // Select this floor before final analysis, based on traceable pilot evidence.
    "minimum_critical_on_time_pdr": null,
    // State the failure that invalidates this baseline, for example unreconciled counters.
    "negative_case": null
  },
  "evidence": {
    // Require manifest, versions, topology, raw events, final counters, and operator notes at minimum.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Freeze the real four-board topology only after upstream RCP and border-router attachment are repeatable.",
      "method": "Label every board, survey one channel and one work area, record placement and power path, then archive actual Thread roles and binary identities.",
      "done_when": "The configuration survives power cycling and its placement, roles, and reachability are archived."
    },
    {
      "path": "/execution",
      "action": "Choose stable workload and duration from pilots.",
      "method": "Run at least three pilots, changing one offered-load factor at a time until counters reconcile and queue behavior is bounded.",
      "done_when": "Durations, streams, seed, and repetitions are frozen before reportable baseline runs."
    },
    {
      "path": "/acceptance",
      "action": "Predeclare service floor and invalidation rules.",
      "method": "Require reconciliation, static treatment, and sufficient raw evidence to classify each run.",
      "done_when": "A later model result is not needed to decide whether baseline data are valid."
    }
  ]
}


Kedua file divalidasi oleh `schemas/experiment.schema.json`. Schema adalah contract machine-facing; field/range pada tabel berikut disalin dari sana, bukan dibuat sebagai variable notebook.

| Field repo | Contoh yang conform secara bentuk — bukan hasil |
|---|---|
| `setup.nodes` | e.g. empat object: gateway ESP32-S3, RCP ESP32-C6, router endpoint ESP32-C6, low-power endpoint ESP32-C6 |
| `setup.thread_channel` | e.g. 15 |
| `setup.placement` | e.g. posisi, orientation, power path, cable, dan baseline block dalam satu string |
| `setup.firmware_reference` | e.g. source/ESP-IDF commits, `sdkconfig` hashes, dan empat binary hashes |
| `execution.warmup_s` | e.g. 30 |
| `execution.measurement_s` | e.g. 300 |
| `execution.cooldown_s` | e.g. 30 |
| `execution.repetitions` | e.g. 5 |
| `execution.seed` | e.g. 12345 |
| `traffic.streams` | e.g. satu critical dan satu telemetry object dari pilot |
| `scenario.event` | e.g. `none` |
| `scenario.at_s` | e.g. 0 |
| `scenario.duration_s` | e.g. 0 |
| `scenario.target` | e.g. no injected disturbance |
| `treatment.mode` | e.g. `baseline` |
| `treatment.candidate_action` | e.g. `none` |
| `treatment.control_profile` | e.g. UNRESOLVED — tetap template sampai resolved profile sungguhan ada |
| `treatment.host_model` | e.g. false |
| `treatment.remote_actuation` | e.g. false |
| `acceptance.counter_reconciliation` | e.g. true setelah item audit dan aggregate reconciliation lulus |
| `acceptance.minimum_critical_on_time_pdr` | e.g. 0.95, dipilih dari pilot sebelum final repetitions |
| `acceptance.negative_case` | e.g. unreconciled lifecycle invalidates the run |
| `evidence.required_artifacts` | e.g. manifest, versions, topology, events, final counters, item audit, operator notes, terminal status |
| `evidence.operator_notes_required` | e.g. true |
| `evidence.topology_photo_required` | e.g. true |

Tiga `_todo` dihapus bersamaan dengan perubahan `state` ke `ready`, bukan ketika satu subsection terlihat rapi. Bila profile registry belum ada, file tetap template dan physical runs tetap pilot/non-reportable.

## Step 14: Pengujian Fisik Baseline End-to-End
### Prosedur Uji Empat Board pada Topologi Lengkap

Meja hardware tetap empat board: S3 gateway, C6 RCP upstream, endpoint A, dan endpoint B. RCP tidak menerima project workload logic. Wi-Fi memakai private AP yang sudah ada; broker dan host berada pada local network.

Urutan satu pilot:

1. Powered hub, adapter, kabel data, UART crossover, dan ground diperiksa tanpa mengubah pin map Week 1.
2. RCP upstream dinyalakan dengan binary/hash yang sudah lulus.
3. Gateway project dinyalakan; provisioning load, Spinel, Thread formation, Wi-Fi, broker, bridge, dan diagnostic readiness muncul berurutan pada log.
4. Endpoint A dan B project dinyalakan; actual role/parent/partition dicatat lagi karena role dapat berubah dari Week 2.
5. Host membuka ready manifest dan run directory sebelum broker connection.
6. Warm-up berjalan tanpa model. Queue high-water dan trace drop tetap dicatat.
7. Measurement menjalankan workload baseline yang sama pada seluruh repetition. Tidak ada load step, placement move, observation pause, atau command.
8. Host raw-records setiap broker message sebelum decode.
9. Cooldown menghentikan release, menguras bounded queue, meminta final counters, dan menjalankan item audit serta aggregate reconciliation.
10. Terminal status ditulis complete, invalid, atau interrupted. Raw evidence tidak dihapus.
11. Satu pilot terpisah sengaja diinterupsi untuk membuktikan recorder tidak menghasilkan false complete.
12. Setelah pilot stabil, repetition count dan service floor dibekukan; baru kemudian final baseline block dijalankan.

| Bukti per repetition | Contoh format — ganti dengan hasil aktual |
|---|---|
| Source/ESP-IDF/upstream identity | e.g. commits dan build versions tercatat |
| Binary SHA-256: gateway/RCP/A/B | e.g. empat digest 64 karakter |
| Actual role A/B | e.g. router / child |
| Parent RLOC16 A/B | e.g. not applicable / 0x1400 |
| Partition ID A/B | e.g. 123456789 / 123456789 |
| `cldt_gateway_runtime_t.thread_rx_queue` high-water | e.g. 7 |
| `cldt_gateway_runtime_t.observation_queue` high-water | e.g. 11 |
| Endpoint queue high-water | e.g. A 4; B 3 |
| Trace drops gateway/A/B | e.g. 0 / 0 / 0 |
| Raw broker records | e.g. 3600 |
| Unique logical items | e.g. 1200 |
| Duplicate terminals | e.g. 0 |
| Unresolved at terminal snapshot | e.g. 0 |
| Aggregate reconciliation | e.g. PASS |
| Item audit | e.g. PASS |
| Terminal status | e.g. complete |
| Operator notes | e.g. no unplanned disturbance |

| Repetition | Raw records | Logical items | Trace drops | Queue high-water | Item audit | Aggregate | Status |
|---:|---:|---:|---:|---:|---|---|---|
| 1 | e.g. 3600 | e.g. 1200 | e.g. 0 | e.g. 7 | e.g. PASS | e.g. PASS | e.g. complete |
| 2 | e.g. 3598 | e.g. 1200 | e.g. 2 | e.g. 9 | e.g. FAIL | e.g. FAIL | e.g. invalid |
| 3 | e.g. 3600 | e.g. 1200 | e.g. 0 | e.g. 8 | e.g. PASS | e.g. PASS | e.g. complete |
| 4 | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run |
| 5 | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run |

## Phase 3 Delivery Checklist
### Sembilan Langkah Implementasi dan Kriteria Penyelesaian (Week 3)

| No. | Pekerjaan | Selesai ketika |
|---:|---|---|
| 1 | tutup contract yang masih ambigu: local terminal, full-queue admission, gateway envelope/topic, NDJSON line, JSON parser, dan profile registry | setiap contract mempunyai satu owner source/header dan test boundary |
| 2 | implement `cldt_protocol_encoded_size()`, `cldt_protocol_encode()`, `cldt_protocol_decode()`, `cldt_protocol_validate_command()`, lalu selesaikan `test_protocol` | fixed bytes dan seluruh mutation case pass |
| 3 | implement `cldt_clock_sync_reset()`, `cldt_clock_sync_observe()`, `cldt_clock_sync_map_to_gateway()`, lalu selesaikan `test_clock_sync` | setiap mapped time membawa uncertainty; one-way claim tetap mati saat mapping invalid |
| 4 | implement `cldt_control_profile_validate()` dan `test_control_profile`; resolve registry hanya dari artefak repo yang nyata | intrinsic profile validation pass; ready manifest tetap blocked bila registry belum ada |
| 5 | implement `cldt_thread_transport_*` serta observation subset `cldt_endpoint_runtime_*` | endpoint mengirim satu exact frame tanpa mengaktifkan command path |
| 6 | implement init/begin/end subset `cldt_policy_guard_*`, `cldt_thread_bridge_*`, tambahkan `thread_diagnostic.c` ke component, lalu implement provisioning/backhaul serta observation subset `cldt_gateway_runtime_*` | safe policy tetap aktif, remote disabled, dua bounded queues punya owner dan clean stop |
| 7 | implement manifest admission, baseline path `main`/coordinator, `cldt_broker_io_*`, serta `cldt_run_recorder_*` | interrupted-run test menyimpan raw record dan satu interrupted status |
| 8 | selesaikan lifecycle subset reproduction serta cross-check dengan `cldt_metrics_audit_sorted_trace()` dan `cldt_metrics_reconcile()` | item audit dan aggregate result dapat diulang dari raw evidence |
| 9 | isi `baseline.JSONC` dari pilot, promote strict JSON hanya bila eligible, lalu jalankan baseline repetitions | setiap run complete/invalid/interrupted dan raw evidence tetap utuh |

### Closure Week 3

- [ ] `test_protocol` pass dengan fixed bytes dan mutation cases lengkap.
- [ ] `test_clock_sync` pass, atau seluruh one-way timing claim dinonaktifkan dan alasan itu tertulis pada evidence.
- [ ] `test_control_profile` pass; profile ID/digest hanya berasal dari registry artefak yang nyata.
- [ ] `test_auth` boleh tetap skip; command path dan remote actuation masih disabled.
- [ ] Project frame bergerak endpoint → gateway → broker → host tanpa perubahan raw bytes yang tidak terdokumentasi.
- [ ] `endpoint_runtime.c` mempertahankan local-accounting regression dan tidak menjalankan `cldt_endpoint_runtime_receive_command()` pada baseline.
- [ ] `policy_guard` memulai baseline pada safe policy dengan remote actuation disabled; accept/fallback path belum digunakan.
- [ ] `thread_rx_queue` dan `observation_queue` bounded, mempunyai owner, high-water, dan visible overflow.
- [ ] `thread_diagnostic.c` benar-benar ikut build dan poll hanya melalui OpenThread lock yang benar.
- [ ] Recorder membuat direktori baru, membekukan manifest, dan menulis `events.NDJSON` append-only.
- [ ] Intentional interruption menghasilkan status interrupted, bukan complete.
- [ ] Setiap logical item lulus item audit atau run ditandai invalid.
- [ ] Aggregate reconciliation lulus per class atau run ditandai invalid.
- [ ] Baseline repetitions memakai topology, firmware, channel, workload, dan placement yang sama.
- [ ] Host model, Kalman, fidelity gate, command authentication, policy, power, SMP, dashboard, dan context shift belum diaktifkan.



> **Akhir Week 3 yang sah:** raw observation path dapat diulang dan seluruh lifecycle dapat dijelaskan dari evidence. Grafik model belum diperlukan. Bila exact envelope, recorder line, atau profile registry belum selesai, baseline tetap pilot dan manifest tetap template—bukan dibuat hijau dengan nama atau nilai fiktif.